In [ ]:
import re
import os
import json
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Dict, List, Tuple, Optional, Any

from scipy.stats import pearsonr, spearmanr, norm
from sklearn.metrics.pairwise import cosine_distances

import sys
sys.path.append("/Users/aniluchavez/Documents/Language/Python")
from spike_processing_utils import (
    load_mat_data,
    get_cells_by_region,
    extract_speaker_events,
    compute_spike_sums,
)

TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def get_speaker_cols(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r"Speaker\d+", str(c))]
    return sorted(cols, key=lambda x: int(x.replace("Speaker", "")))

def cell_to_first_token(x):
    if pd.isna(x):
        return np.nan
    toks = TOKEN_RE.findall(str(x).strip().lower())
    return toks[0] if toks else np.nan

def words_in_extraction_order(df: pd.DataFrame, speaker_col: str) -> List[Any]:
    tmp = df.copy()
    tmp[speaker_col] = tmp[speaker_col].astype(str).str.strip()
    spk_df = tmp[tmp[speaker_col] != ""]
    return [cell_to_first_token(x) for x in spk_df[speaker_col].values]

def make_shared_counts(words_self: List[str], words_other: List[str]):
    c_self = Counter(words_self)
    c_other = Counter(words_other)
    shared = sorted(set(c_self.keys()) & set(c_other.keys()))
    counts_df = pd.DataFrame({
        "word": shared,
        "n_self": [c_self[w] for w in shared],
        "n_other": [c_other[w] for w in shared],
        "n_total": [c_self[w] + c_other[w] for w in shared],
    }).sort_values(["n_total", "word"], ascending=[False, True]).reset_index(drop=True)
    return shared, counts_df

def word_word_cosine_distance(FR: np.ndarray, col_labels: List[str], fill="col_mean") -> pd.DataFrame:
    X = FR.astype(float).copy()
    if np.isnan(X).any():
        if fill == "col_mean":
            col_means = np.nanmean(X, axis=0, keepdims=True)
            X = np.where(np.isfinite(X), X, col_means)
        elif fill == "zero":
            X = np.nan_to_num(X, nan=0.0)
        else:
            raise ValueError("fill must be 'col_mean' or 'zero'")
    D = cosine_distances(X.T)
    return pd.DataFrame(D, index=col_labels, columns=col_labels)

def vectorize_upper_triangle(D: pd.DataFrame) -> np.ndarray:
    iu = np.triu_indices(D.shape[0], k=1)
    return D.values[iu]

def fisher_required_n_pairs(r_effect: float, alpha: float, power: float) -> int:
    # heuristic “effective N pairs” power gate
    if not (0 < abs(r_effect) < 1):
        return int(1e18)
    z_effect = np.arctanh(r_effect)
    z_alpha = norm.ppf(1 - alpha / 2)
    z_power = norm.ppf(power)
    n_minus_3 = ((z_alpha + z_power) / abs(z_effect)) ** 2
    return int(np.ceil(n_minus_3 + 3))

def geometry_permutation_test(D_self: pd.DataFrame, D_other: pd.DataFrame,
                              n_perm=5000, method="spearman", seed=0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    words = D_self.index.to_numpy()
    iu = np.triu_indices(len(words), k=1)
    v_self = D_self.values[iu]
    perm_stats = np.zeros(n_perm, dtype=float)

    for i in range(n_perm):
        perm = rng.permutation(len(words))
        Dp = D_other.values[perm][:, perm]
        v_other = Dp[iu]
        mask = np.isfinite(v_self) & np.isfinite(v_other)

        if method == "pearson":
            r, _ = pearsonr(v_self[mask], v_other[mask])
        elif method == "spearman":
            r, _ = spearmanr(v_self[mask], v_other[mask])
        else:
            raise ValueError("method must be 'pearson' or 'spearman'")
        perm_stats[i] = r
    return perm_stats

def perm_p_right_tail(r_obs: float, r_perm: np.ndarray) -> float:
    return (np.sum(r_perm >= r_obs) + 1) / (len(r_perm) + 1)

import os
import matplotlib.pyplot as plt
import numpy as np

def plot_geometry_scatter(v_self, v_other, r_p, r_s, title_prefix="Geometry similarity",
                          save_path=None, show=True):
    plt.figure(figsize=(5, 5))
    plt.scatter(v_self, v_other, s=10, alpha=0.4)
    plt.xlabel("SELF word–word distance")
    plt.ylabel("OTHER word–word distance")
    plt.title(f"{title_prefix}\nPearson r={r_p:.2f}, Spearman ρ={r_s:.2f}")
    # y=x reference
    try:
        plt.axline((0, 0), slope=1, linestyle="--", linewidth=1)
    except Exception:
        # fallback for older matplotlib
        lims = [
            np.nanmin([plt.xlim()[0], plt.ylim()[0]]),
            np.nanmax([plt.xlim()[1], plt.ylim()[1]])
        ]
        plt.plot(lims, lims, linestyle="--", linewidth=1)
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=200)
    if show:
        plt.show()
    plt.close()


def plot_permutation_hist(perm_stats, r_obs, method_label="Spearman",
                          title_prefix="Permutation test: SELF vs OTHER geometry",
                          save_path=None, show=True):
    plt.figure(figsize=(6, 4))
    plt.hist(perm_stats, bins=40, alpha=0.7, label="Null (permuted)")
    plt.axvline(r_obs, linewidth=2, label="Observed")
    plt.xlabel(f"{method_label} geometry correlation")
    plt.ylabel("Count")
    plt.title(title_prefix)
    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=200)
    if show:
        plt.show()
    plt.close()


In [ ]:


from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any
from collections import Counter
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# assumes you already defined/imported:
# get_speaker_cols, words_in_extraction_order, make_shared_counts,
# word_word_cosine_distance, vectorize_upper_triangle,
# fisher_required_n_pairs, geometry_permutation_test, perm_p_right_tail
# and imported:
# load_mat_data, get_cells_by_region, extract_speaker_events, compute_spike_sums


def stability_gate(summary, min_prob_gt_0=0.90, max_ci_width=0.25):
    ok = (summary["prob_r_gt_0"] >= min_prob_gt_0) and (summary["ci_width"] <= max_ci_width)
    reason = f"prob_r_gt_0={summary['prob_r_gt_0']:.2f}, ci_width={summary['ci_width']:.2f}"
    return ok, reason


@dataclass
class RSAConfig:
    patient_id: str
    excel_path: str
    mat_path: str
    region_ranges: Dict[str, List[Tuple[int, int]]]
    region_name: str

    speaker1_col: str = "Speaker1"
    average_repeats: bool = True
    min_repeats_each_side: int = 1
    bin_mode: str = "explicit_event_bounds"

    window_spec_self: Optional[Dict[str, Any]] = None
    window_spec_other: Optional[Dict[str, Any]] = None

    # Heuristic power gate settings
    min_neurons: int = 20
    min_shared_words: int = 20
    power_alpha: float = 0.05
    power_target: float = 0.80
    power_detectable_r: float = 0.20


class RSAAnalyzerNotebook:
    def __init__(self, n_perm=5000, seed=0, verbose=True):
        self.n_perm = int(n_perm)
        self.seed = int(seed)
        self.verbose = bool(verbose)

    def default_window_spec(self):
        return dict(
            speaker_of_interest="Speaker1",
            mode="target_vs_other_fixed_window_from_ref",
            target_ref_point="onset",
            target_shift=250,
            target_window_length=500,
            other_ref_point="offset",
            other_shift=-500,
            other_window_length=300,
        )

    def shared_word_set(self, df: pd.DataFrame, speaker1_col: str, min_repeats_each_side: int):
        speaker_cols = get_speaker_cols(df)
        other_cols = [c for c in speaker_cols if c != speaker1_col]

        spk1_words = [w for w in words_in_extraction_order(df, speaker1_col) if pd.notna(w)]
        other_words = []
        for spk in other_cols:
            other_words.extend([w for w in words_in_extraction_order(df, spk) if pd.notna(w)])

        c1 = Counter(spk1_words)
        co = Counter(other_words)

        shared = set(c1.keys()) & set(co.keys())
        shared = {w for w in shared if (c1[w] >= min_repeats_each_side and co[w] >= min_repeats_each_side)}
        return shared, speaker_cols

    def build_FR(self, cfg: RSAConfig):
        ws_self = cfg.window_spec_self or self.default_window_spec()
        ws_other = cfg.window_spec_other or self.default_window_spec()

        df = pd.read_excel(cfg.excel_path, keep_default_na=False)
        shared, speaker_cols = self.shared_word_set(df, cfg.speaker1_col, cfg.min_repeats_each_side)
        other_cols = [c for c in speaker_cols if c != cfg.speaker1_col]

        spikes, qual, chan = load_mat_data(cfg.mat_path)
        region_cells_all = get_cells_by_region(chan, qual, cfg.region_ranges)
        if cfg.region_name not in region_cells_all:
            raise ValueError(f"[{cfg.patient_id}] Region '{cfg.region_name}' not found")

        region_cells = {cfg.region_name: region_cells_all[cfg.region_name]}
        n_neurons = len(region_cells_all[cfg.region_name])
        neuron_ids = list(range(n_neurons))

        events_self_all = extract_speaker_events(cfg.excel_path, **ws_self)
        events_other_all = extract_speaker_events(cfg.excel_path, **ws_other)

        ev_self = events_self_all.get(cfg.speaker1_col)
        if ev_self is None or len(ev_self) == 0:
            raise ValueError(f"[{cfg.patient_id}] No {cfg.speaker1_col} events")

        words_self_full = words_in_extraction_order(df, cfg.speaker1_col)[: ev_self.shape[0]]
        mask_self = np.array([(w in shared) for w in words_self_full], dtype=bool)
        ev_self = ev_self[mask_self]
        words_self = np.array(words_self_full, dtype=object)[mask_self]

        ev_other_list, words_other_list = [], []
        for spk in other_cols:
            ev = events_other_all.get(spk)
            if ev is None or len(ev) == 0:
                continue
            words_full = words_in_extraction_order(df, spk)[: ev.shape[0]]
            mask = np.array([(w in shared) for w in words_full], dtype=bool)
            if np.any(mask):
                ev_other_list.append(ev[mask])
                words_other_list.append(np.array(words_full, dtype=object)[mask])

        if not ev_other_list:
            raise ValueError(f"[{cfg.patient_id}] No shared-word events for other speakers")

        ev_other = np.vstack(ev_other_list)
        words_other = np.concatenate(words_other_list)

        col_labels, shared_counts_df = make_shared_counts(
            [w for w in words_self.tolist() if pd.notna(w)],
            [w for w in words_other.tolist() if pd.notna(w)],
        )

        def compute_fr(ev):
            sums = compute_spike_sums(spikes, region_cells, ev, mode=cfg.bin_mode)
            mat = sums[cfg.region_name]  # [events x neurons]
            win_ms = ev[:, 2] - ev[:, 1]
            win_s = np.where(win_ms > 0, win_ms / 1000.0, np.nan)
            return mat / win_s[:, None]

        fr_self_events = compute_fr(ev_self)    # [Eself x N]
        fr_other_events = compute_fr(ev_other)  # [Eother x N]

        if not cfg.average_repeats:
            return fr_self_events.T, fr_other_events.T, col_labels, neuron_ids, shared_counts_df

        FR_self = np.full((n_neurons, len(col_labels)), np.nan)
        FR_other = np.full((n_neurons, len(col_labels)), np.nan)
        for j, w in enumerate(col_labels):
            ii = np.where(words_self == w)[0]
            jj = np.where(words_other == w)[0]
            if ii.size:
                FR_self[:, j] = np.nanmean(fr_self_events[ii, :], axis=0)
            if jj.size:
                FR_other[:, j] = np.nanmean(fr_other_events[jj, :], axis=0)

        return FR_self, FR_other, col_labels, neuron_ids, shared_counts_df

    def power_gate(self, cfg: RSAConfig, n_neurons: int, n_words: int):
        if n_neurons < cfg.min_neurons:
            return False, f"Too few neurons: {n_neurons} < {cfg.min_neurons}"
        if n_words < cfg.min_shared_words:
            return False, f"Too few shared words: {n_words} < {cfg.min_shared_words}"

        n_pairs = n_words * (n_words - 1) // 2
        n_req = fisher_required_n_pairs(cfg.power_detectable_r, cfg.power_alpha, cfg.power_target)
        if n_pairs < n_req:
            return False, f"Underpowered (heuristic): n_pairs={n_pairs} < n_req≈{n_req} for r={cfg.power_detectable_r}"
        return True, "OK"

    def _rsa_corr_from_FR(self, FR_self, FR_other, col_labels, method="spearman"):
        D_self  = word_word_cosine_distance(FR_self, col_labels)
        D_other = word_word_cosine_distance(FR_other, col_labels)

        v_self  = vectorize_upper_triangle(D_self)
        v_other = vectorize_upper_triangle(D_other)

        mask = np.isfinite(v_self) & np.isfinite(v_other)
        v_self, v_other = v_self[mask], v_other[mask]

        if method == "pearson":
            r, p = pearsonr(v_self, v_other)
        elif method == "spearman":
            r, p = spearmanr(v_self, v_other)
        else:
            raise ValueError("method must be 'pearson' or 'spearman'")
        return float(r), float(p)

    def bootstrap_rsa_stability(
        self,
        FR_self: np.ndarray,
        FR_other: np.ndarray,
        col_labels: List[str],
        method: str = "spearman",
        n_boot: int = 1000,
        word_frac: float = 0.80,
        neuron_frac: float = 1.00,
        seed: int = 0,
    ):
        rng = np.random.default_rng(seed)
        n_neurons, n_words = FR_self.shape

        w_k = max(3, int(np.ceil(word_frac * n_words)))
        n_k = max(3, int(np.ceil(neuron_frac * n_neurons)))

        rs = np.empty(n_boot, dtype=float)

        word_idx_all = np.arange(n_words)
        neuron_idx_all = np.arange(n_neurons)

        for b in range(n_boot):
            w_idx = rng.choice(word_idx_all, size=w_k, replace=False)
            n_idx = rng.choice(neuron_idx_all, size=n_k, replace=True)

            FRs = FR_self[n_idx][:, w_idx]
            FRo = FR_other[n_idx][:, w_idx]
            labels_sub = [col_labels[i] for i in w_idx]

            r, _ = self._rsa_corr_from_FR(FRs, FRo, labels_sub, method=method)
            rs[b] = r

        ci_lo, ci_hi = np.nanpercentile(rs, [2.5, 97.5])
        summary = dict(
            method=method,
            n_boot=int(n_boot),
            word_frac=float(word_frac),
            neuron_frac=float(neuron_frac),
            r_mean=float(np.nanmean(rs)),
            r_median=float(np.nanmedian(rs)),
            ci95=(float(ci_lo), float(ci_hi)),
            ci_width=float(ci_hi - ci_lo),
            prob_r_gt_0=float(np.mean(rs > 0)),
            prob_r_gt_0p1=float(np.mean(rs > 0.10)),
            prob_r_gt_0p2=float(np.mean(rs > 0.20)),
        )
        return rs, summary

    def run(self, cfg: RSAConfig, plot=False, show_plots=True,
        save_figs=False, out_dir=None):
        FR_self, FR_other, col_labels, neuron_ids, shared_counts_df = self.build_FR(cfg)

        n_neurons = FR_self.shape[0]
        n_words = len(col_labels)
        n_pairs = n_words * (n_words - 1) // 2

        powered, reason = self.power_gate(cfg, n_neurons, n_words)

        # distances
        D_self = word_word_cosine_distance(FR_self, col_labels)
        D_other = word_word_cosine_distance(FR_other, col_labels)

        v_self = vectorize_upper_triangle(D_self)
        v_other = vectorize_upper_triangle(D_other)
        mask = np.isfinite(v_self) & np.isfinite(v_other)
        v_self, v_other = v_self[mask], v_other[mask]

        r_p, p_p = pearsonr(v_self, v_other)
        r_s, p_s = spearmanr(v_self, v_other)

        # permutation nulls
        perm_p = geometry_permutation_test(D_self, D_other, n_perm=self.n_perm, method="pearson", seed=self.seed)
        perm_s = geometry_permutation_test(D_self, D_other, n_perm=self.n_perm, method="spearman", seed=self.seed)

        perm_pval_p = perm_p_right_tail(r_p, perm_p)
        perm_pval_s = perm_p_right_tail(r_s, perm_s)

        # stability diagnostics
        rs_word, summ_word = self.bootstrap_rsa_stability(
            FR_self, FR_other, col_labels, method="spearman",
            n_boot=1000, word_frac=0.80, neuron_frac=1.00, seed=self.seed
        )
        rs_neur, summ_neur = self.bootstrap_rsa_stability(
            FR_self, FR_other, col_labels, method="spearman",
            n_boot=1000, word_frac=1.00, neuron_frac=1.00, seed=self.seed + 1
        )

        word_stable, word_reason = stability_gate(summ_word)
        neur_stable, neur_reason = stability_gate(summ_neur)

        out = dict(
            patient_id=cfg.patient_id,
            region=cfg.region_name,
            n_neurons=n_neurons,
            n_shared_words=n_words,
            n_pairs=int(n_pairs),

            powered_enough=powered,
            power_reason=reason,

            pearson_r=float(r_p),
            pearson_p=float(p_p),
            spearman_r=float(r_s),
            spearman_p=float(p_s),

            perm_p_pearson=float(perm_pval_p),
            perm_p_spearman=float(perm_pval_s),

            word_boot_ci95=summ_word["ci95"],
            word_boot_ci_width=summ_word["ci_width"],
            word_prob_r_gt_0=summ_word["prob_r_gt_0"],
            word_stable=word_stable,
            word_stability_reason=word_reason,

            neuron_boot_ci95=summ_neur["ci95"],
            neuron_boot_ci_width=summ_neur["ci_width"],
            neuron_prob_r_gt_0=summ_neur["prob_r_gt_0"],
            neuron_stable=neur_stable,
            neuron_stability_reason=neur_reason,
        )

        # ---- plotting / saving ----
        if out_dir is None and save_figs:
            # default: save into a folder next to the notebook
            out_dir = "./rsa_outputs"

        if save_figs and out_dir is not None:
            fig_dir = os.path.join(out_dir, cfg.patient_id, cfg.region_name)
        else:
            fig_dir = None

        if plot:
            # scatter
            scatter_path = None if fig_dir is None else os.path.join(fig_dir, f"{cfg.patient_id}_{cfg.region_name}_scatter.png")
            plot_geometry_scatter(
                v_self, v_other, r_p, r_s,
                title_prefix=f"{cfg.patient_id} {cfg.region_name}",
                save_path=scatter_path,
                show=show_plots
            )

            # permutation hist (spearman)
            hist_path = None if fig_dir is None else os.path.join(fig_dir, f"{cfg.patient_id}_{cfg.region_name}_perm_spearman.png")
            plot_permutation_hist(
                perm_s, r_s, method_label="Spearman",
                title_prefix=f"{cfg.patient_id} {cfg.region_name} permutation (Spearman)",
                save_path=hist_path,
                show=show_plots
            )

        if self.verbose:
            print(
                f"[{cfg.patient_id} | {cfg.region_name}] neurons={n_neurons}, words={n_words}, pairs={n_pairs}\n"
                f"  Spearman={r_s:.3f} (perm p={perm_pval_s:.3g}) | Pearson={r_p:.3f} (perm p={perm_pval_p:.3g})\n"
                f"  Heuristic power={powered} ({reason})\n"
                f"  Word stability={word_stable} ({word_reason}) | Neuron stability={neur_stable} ({neur_reason})"
            )

        return out, D_self, D_other, shared_counts_df, rs_word, rs_neur


In [ ]:
region_ranges = {"hippocampus": [(1,16),(25,40)]}

window_self = dict(
    speaker_of_interest="Speaker1",
    mode="target_vs_other_custom_bounds",
    target_start_ref="onset",
    target_start_shift=-200,
    target_end_ref="offset",
    target_end_shift=-200,
    other_start_ref="onset",
    other_start_shift=200,
    other_end_ref="offset",
    other_end_shift=200,
)


window_other = window_self.copy()

cfg = RSAConfig(
    patient_id="YEU",
    excel_path="/Users/aniluchavez/Documents/Language/YEU/147/20250423_PTYEU_task147_convoFinalwithPuncta_withPOS.xlsx",
    mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YEU/ptYEU_task147_new_spikes.mat",
    region_ranges=region_ranges,
    region_name="hippocampus",
    window_spec_self=window_self,
    window_spec_other=window_other,
    min_neurons=10,
    min_shared_words=10,
    power_detectable_r=0.20,
)

an = RSAAnalyzerNotebook(n_perm=5000, seed=0, verbose=True)
out, D_self, D_other, shared_counts_df, rs_word, rs_neur = an.run(cfg)

out


In [ ]:
# window_self = dict(
#     speaker_of_interest="Speaker1",
#     mode="target_vs_other_custom_bounds",
#     target_start_ref="onset",
#     target_start_shift=-200,
#     target_end_ref="offset",
#     target_end_shift=-200,
#     other_start_ref="onset",
#     other_start_shift=200,
#     other_end_ref="offset",
#     other_end_shift=200,
# )
window_self = dict(
    speaker_of_interest="Speaker1",
    mode="target_vs_other_fixed_window_from_ref",
    target_ref_point="onset",
    target_shift=-150,
    target_window_length=500,
    other_ref_point="onset",
    other_shift=200,
    other_window_length=500,
)
window_other = window_self.copy()

configs = [
    RSAConfig(
        patient_id="YEY",
        excel_path="/Users/aniluchavez/Documents/Language/YEY/086/20250714_PTYEY_task86_convo_RoomMic1F1.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YEY/ptYEY_task86_new_spikes.mat",
        region_ranges={"hippocampus": [(1, 16)]},  # <-- YEY-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),
    RSAConfig(
        patient_id="YEV",
        excel_path="/Users/aniluchavez/Documents/Language/YEV/037/PTYEV_task37_convo_RoomMic_MFAFinalWPunct_englishonly_cp.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YEV/ptYEV_task37_new_spikes.mat",
        region_ranges={"hippocampus": [(1,16),(25,40)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),

    RSAConfig(
        patient_id="YEU",
        excel_path="/Users/aniluchavez/Documents/Language/YEU/147/20250423_PTYEU_task147_convoFinalwithPuncta_withPOS.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YEU/ptYEU_task147_new_spikes.mat",
        region_ranges={"hippocampus": [(1,16),(25,40)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),

    RSAConfig(
        patient_id="YFI",
        excel_path="/Users/aniluchavez/Documents/Language/YFI/081/Cut_2/20250324_PTYFI_task81_convoFinalWPunct_half2.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFI/ptYFI_task81_new_spikes.mat",
        region_ranges={"hippocampus": [(1,8),(25,40)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),

    RSAConfig(
        patient_id="YFG",
        excel_path="/Users/aniluchavez/Documents/Language/YFG/PTYFG_task18_convoFinalWPunct_cp.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFG/ptYFG_task18_new_spikes.mat",
        region_ranges={"hippocampus": [(9,16)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),
        RSAConfig(
        patient_id="YFF",
        excel_path="/Users/aniluchavez/Documents/Language/YFF/017/PTYFF_task17_convoFinalWPunct_cp.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFF/ptYFF_task17_new_spikes.mat",
        region_ranges={"hippocampus": [(9,16),(25,40)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),
        RSAConfig(
        patient_id="YFC",
        excel_path="/Users/aniluchavez/Documents/Language/YFC/028/PTYFC_task28_convoFinalWPunct_noES_cp.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFC/ptYFC_task28_new_spikes.mat",
        region_ranges={"hippocampus": [(1,8),(33,48)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),
        RSAConfig(
        patient_id="YEZ",
        excel_path="/Users/aniluchavez/Documents/Language/YEZ/060/20250419_PTYEZ_task60_convo_RoomMic1_eonlywpunct.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YEZ/ptYEZ_task60_new_spikes.mat",
        region_ranges={"hippocampus": [(1,16)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),
        RSAConfig(
        patient_id="YFK",
        excel_path="/Users/aniluchavez/Documents/Language/YFK/040/PTYFK_task40_convo_mfa_FINAL_FIXED220250528.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFK/ptYFK_task40_new_spikes.mat",
        region_ranges={"hippocampus": [(1,16),(25,40)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),
        RSAConfig(
        patient_id="YFA",
        excel_path="/Users/aniluchavez/Documents/Language/YFA/PTYFA_task25_convo_withpuncta_cp.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFA/ptYFA_task25_new_spikes.mat",
        region_ranges={"hippocampus": [(1,16),(25,40)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),
        RSAConfig(
        patient_id="YFM",
        excel_path="/Users/aniluchavez/Documents/Language/YFM/PTYFM_task104_filtered_used_rows_withNP_withClusterIDNew.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFM/ptYFM_task104_new_spikes.mat",
        region_ranges={"hippocampus": [(33,48)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),

        RSAConfig(
        patient_id="YFP",
        excel_path="/Users/aniluchavez/Documents/Language/YFP/PTYFP_task88_filtered_used_rows_withNP_withClusterIDNew.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFP/ptYFP_task88_new_spikes.mat",
        region_ranges={"hippocampus": [(17,24),(25,32),(49,56),(57,64)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),

        RSAConfig(
        patient_id="YFS",
        excel_path="/Users/aniluchavez/Documents/Language/YFS/PTYFS_task95_filtered_used_rows_withNP_withClusterIDNew.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFS/ptYFS_task95_new_spikes.mat",
        region_ranges={"hippocampus": [(1,24)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),

        RSAConfig(
        patient_id="YFR",
        excel_path="/Users/aniluchavez/Documents/Language/YFR/PTYFR_task91_filtered_used_rows_withNP_withClusterIDNew.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFR/ptYFR_task91_new_spikes.mat",
        region_ranges={"hippocampus": [(1,16),(41,56)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),
        RSAConfig(
        patient_id="YFU",
        excel_path="/Users/aniluchavez/Documents/Language/YFU/PTYFU_task224_filtered_used_rows_withNP_withClusterIDNew.xlsx",
        mat_path="/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFU/ptYFU_task224_new_spikes.mat",
        region_ranges={"hippocampus": [(17,32),(41,56)]},  # <-- YEV-specific
        region_name="hippocampus",
        window_spec_self=window_self,
        window_spec_other=window_other,
    ),
]
an = RSAAnalyzerNotebook(n_perm=5000, seed=0, verbose=True)

rows = []
for cfg in configs:
    out, *_ = an.run(cfg, plot=True, show_plots=False, save_figs=True, out_dir="./rsa_outputs")
    rows.append(out)

summary_df = pd.DataFrame(rows)
summary_df.to_excel("./rsa_outputs/multipatient_rsa_summary.xlsx", index=False)


In [ ]:
summary_df.to_excel("./rsa_outputs/multipatient_rsa_summary.xlsx", index=False)


In [ ]:
pval_df = summary_df[["patient_id", "region", "perm_p_spearman"]].copy()
pval_df


In [ ]:
import matplotlib as mpl

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Helvetica"]
mpl.rcParams["font.size"] = 12

# VERY IMPORTANT for Illustrator editable text
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["pdf.fonttype"] = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp

def mean_ci(x, ci=95):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    mean = np.mean(x)
    if n < 2:
        return mean, np.nan, np.nan
    se = np.std(x, ddof=1) / np.sqrt(n)
    z = 1.96
    lo = mean - z * se
    hi = mean + z * se
    return mean, lo, hi

def p_to_stars(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

def plot_patient_rsa_barplot(
    summary_df,
    metric="spearman_r",
    p_metric="perm_p_spearman",
    title=None,
    ylabel=None,
    sort=True,
    sig_only=False,
    sig_threshold=0.05,
    figsize=(7, 5),
    save_path=None,
    show=True,
    color="#9167EE",
    bar_width=0.4,
    spacing=0.5,
):
    df = summary_df.copy()
    df = df[np.isfinite(df[metric])].copy()
    if sig_only and p_metric in df.columns:
        df = df[df[p_metric] < sig_threshold].copy()

    if sort:
        df = df.sort_values(metric, ascending=False).reset_index(drop=True)

    vals = df[metric].values

    labels = [
        f"{pid}\n(n={n})"
        for pid, n in zip(df["patient_id"], df["n_neurons"])
    ]

    x = np.arange(len(df)) * spacing
    group_mean, ci_lo, ci_hi = mean_ci(vals, ci=95)

    if len(vals) >= 2:
        _, p_group = ttest_1samp(vals, 0.0, nan_policy="omit")
    else:
        p_group = np.nan

    fig, ax = plt.subplots(figsize=figsize)

    ax.bar(
        x,
        vals,
        width=bar_width,
        alpha=0.9,
        color=color,
        edgecolor="black",
        linewidth=0.8,
    )

    # significance stars by p-value
    if p_metric in df.columns:
        for xi, yi, pp in zip(x, vals, df[p_metric].values):
            stars = p_to_stars(pp)
            if stars:
                y_text = yi + 0.02 if yi >= 0 else yi - 0.05
                va = "bottom" if yi >= 0 else "top"
                ax.text(
                    xi, y_text, stars,
                    ha="center", va=va,
                    fontsize=12
                )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", rotation_mode="anchor")
    ax.set_ylabel(ylabel or metric)
    ax.set_xlabel("patient")
    ax.set_title(title or f"RSA across patients ({metric})\nGroup p vs 0 = {p_group:.3g}", pad=16)
    ax.set_xlim(x.min() - spacing, x.max() + spacing)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()

    return {
        "n_patients": int(len(vals)),
        "group_mean": float(group_mean),
        "ci95_lo": float(ci_lo) if np.isfinite(ci_lo) else np.nan,
        "ci95_hi": float(ci_hi) if np.isfinite(ci_hi) else np.nan,
        "group_p_vs_0": float(p_group) if np.isfinite(p_group) else np.nan,
    }

In [ ]:
stats_s = plot_patient_rsa_barplot(
    summary_df,
    metric="spearman_r",
    p_metric="perm_p_spearman",
    title="speaking and listening geometry similarity across patients",
    ylabel="spearman rsa",
    save_path="./rsa_outputs/group_barplot_spearman.eps",
    show=True,
    sig_only=False,
)

stats_p = plot_patient_rsa_barplot(
    summary_df,
    metric="pearson_r",
    p_metric="perm_p_pearson",
    title="SELF vs OTHER geometry similarity across patients",
    ylabel="Pearson RSA",
    save_path="./rsa_outputs/group_barplot_pearson.eps",
    show=True,
    sig_only=False,
)

print(stats_s)
print(stats_p)

# start of rdm analysis, first we build the firing rate matrices words by neuron

In [ ]:
def build_word_neuron_mats(analyzer, cfg):
    """
    Uses analyzer.build_FR(cfg) which returns FR_self/FR_other as (neurons x words).
    Converts to (words x neurons) for RDM analysis.
    """
    FR_self, FR_other, words, neuron_ids, shared_counts_df = analyzer.build_FR(cfg)

    X_self = FR_self.T    # words x neurons
    X_other = FR_other.T  # words x neurons

    # handy combined option: mean across conditions per word/neuron
    X_mean = np.nanmean(np.stack([X_self, X_other], axis=0), axis=0)

    return {
        "words": list(words),
        "neuron_ids": list(neuron_ids),
        "shared_counts_df": shared_counts_df,
        "X_self": X_self,
        "X_other": X_other,
        "X_mean": X_mean,
    }

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform

def rdm_kernel_gram(X_words_by_neurons: np.ndarray):
    """
    X_words_by_neurons: (n_words x n_neurons)

    Returns dict with:
      RDM: cosine distance matrix
      K:   similarity kernel (1 - RDM), symmetrized
      J:   centering matrix
      G:   centered Gram = J K J
    """
    X = np.asarray(X_words_by_neurons, dtype=float)

    # Optional: handle NaNs robustly (should be rare with your averaging, but safe)
    if np.isnan(X).any():
        col_means = np.nanmean(X, axis=0, keepdims=True)
        X = np.where(np.isfinite(X), X, col_means)

    n = X.shape[0]
    RDM = squareform(pdist(X, metric="cosine"))
    K = 1.0 - RDM
    K = 0.5 * (K + K.T)

    J = np.eye(n) - np.ones((n, n)) / n
    G = J @ K @ J

    return {"RDM": RDM, "K": K, "J": J, "G": G}

In [ ]:
def run_rdm_for_cfg(analyzer, cfg):
    mats = build_word_neuron_mats(analyzer, cfg)

    out = dict(
        patient_id=cfg.patient_id,
        region=cfg.region_name,
        words=mats["words"],
        neuron_ids=mats["neuron_ids"],
        shared_counts_df=mats["shared_counts_df"],
        X_self=mats["X_self"],
        X_other=mats["X_other"],
        X_mean=mats["X_mean"],
    )

    out["self"]  = rdm_kernel_gram(mats["X_self"])
    out["other"] = rdm_kernel_gram(mats["X_other"])
    out["mean"]  = rdm_kernel_gram(mats["X_mean"])

    return out

# runs first patient (or 1 patient)

In [ ]:
an = RSAAnalyzerNotebook(verbose=False)

rdm_out = run_rdm_for_cfg(an, cfg)
G_self = rdm_out["self"]["G"]
G_other = rdm_out["other"]["G"]
rdm_out

# runs all patients

In [ ]:
def run_rdm_all(analyzer, configs):
    out_dict = {}
    for cfg in configs:
        out_dict[cfg.patient_id] = out_dict.get(cfg.patient_id, {})
        out_dict[cfg.patient_id][cfg.region_name] = run_rdm_for_cfg(analyzer, cfg)
        print(f"[{cfg.patient_id}] words={len(out_dict[cfg.patient_id][cfg.region_name]['words'])}, "
              f"neurons={out_dict[cfg.patient_id][cfg.region_name]['X_self'].shape[1]}")
    return out_dict

In [ ]:
rdm_dict = run_rdm_all(an, configs)

In [ ]:
import numpy as np

def eig_psd(G: np.ndarray, eps: float = 1e-10, sort_desc: bool = True):
    """
    PSD-guarded eigendecomposition for symmetric Gram matrix.

    Returns:
      V : (n x n) eigenvectors (columns)
      evals : (n,) eigenvalues (sorted desc by default, clipped to >=0)
    """
    G = np.asarray(G, dtype=float)
    G = 0.5 * (G + G.T)  # symmetrize

    # symmetric eigendecomposition
    evals, V = np.linalg.eigh(G)  # ascending by default

    # clip small negatives (numerical)
    evals = np.where(evals < eps, 0.0, evals)

    if sort_desc:
        idx = np.argsort(evals)[::-1]
        evals = evals[idx]
        V = V[:, idx]

    return V, evals

def participation_ratio(evals: np.ndarray, eps: float = 1e-12) -> float:
    """
    Effective dimensionality / participation ratio:
      (sum λ)^2 / sum λ^2
    """
    lam = np.asarray(evals, dtype=float)
    lam = lam[lam > eps]
    if lam.size == 0:
        return 0.0
    return float((lam.sum() ** 2) / np.sum(lam ** 2))

def k95_from_evals(evals: np.ndarray, frac: float = 0.95, eps: float = 1e-12) -> int:
    """
    Smallest k such that cumulative explained variance >= frac.
    Assumes evals are nonnegative and sorted descending.
    """
    lam = np.asarray(evals, dtype=float)
    total = lam.sum()
    if total <= eps:
        return 0
    ve = lam / total
    return int(np.searchsorted(np.cumsum(ve), frac) + 1)  # +1 for 1-indexed k

def attach_eigendecomp_to_rdm_out(rdm_out, frac=0.95, cap=150, eps=1e-10, store_full_V=False):
    """
    Adds an 'eig' field into rdm_out with eigenvalues/vectors + dim stats.
    store_full_V=False saves memory (stores only U=top-k).
    """
    Gs = rdm_out["self"]["G"]
    Go = rdm_out["other"]["G"]

    Vs, es = eig_psd(Gs, eps=eps, sort_desc=True)
    Vo, eo = eig_psd(Go, eps=eps, sort_desc=True)

    eff_s = participation_ratio(es)
    eff_o = participation_ratio(eo)

    k95_s = k95_from_evals(es, frac=frac)
    k95_o = k95_from_evals(eo, frac=frac)

    k = int(min(k95_s, k95_o, cap))

    eig_block = dict(
        frac=float(frac),
        cap=int(cap),
        eps=float(eps),

        evals_self=es,
        evals_other=eo,

        eff_dim_self=float(eff_s),
        eff_dim_other=float(eff_o),

        k95_self=int(k95_s),
        k95_other=int(k95_o),

        k=int(k),

        U_self=Vs[:, :k],
        U_other=Vo[:, :k],

        sum_topk_eigs_self=float(es[:k].sum()) if k > 0 else 0.0,
        sum_topk_eigs_other=float(eo[:k].sum()) if k > 0 else 0.0,
    )

    if store_full_V:
        eig_block["V_self"] = Vs
        eig_block["V_other"] = Vo

    rdm_out["eig"] = eig_block
    return rdm_out

# for one

In [ ]:
rdm_out = attach_eigendecomp_to_rdm_out(rdm_out, frac=0.95, cap=150)

rdm_out["eig"]["eff_dim_self"], rdm_out["eig"]["k95_self"], rdm_out["eig"]["k"]
rdm_out["eig"]["U_self"].shape, rdm_out["eig"]["U_other"].shape

# for all

In [ ]:
for pid, regions in rdm_dict.items():
    for region, rdm_out in regions.items():
        attach_eigendecomp_to_rdm_out(rdm_out, frac=0.95, cap=150)
        print(pid, region, rdm_out["eig"]["k"])

In [ ]:
import numpy as np
from scipy.linalg import subspace_angles

def principal_angles_deg(Ua: np.ndarray, Ub: np.ndarray) -> np.ndarray:
    """
    Returns principal angles in degrees between column spaces of Ua and Ub.
    Ua, Ub: (n_items x k) with orthonormal columns (or close).
    """
    ang = subspace_angles(Ua, Ub)  # radians, sorted ascending
    return np.degrees(ang)

In [ ]:
def elsayed_alignment_var(G: np.ndarray, U: np.ndarray, sum_topk_eigs: float) -> float:
    """
    Directional Elsayed-style alignment:
      tr(U^T G U) / sum_{i<=k} eig_i(G)

    G: (n_words x n_words) centered Gram
    U: (n_words x k) orthonormal basis (e.g., top-k eigenvectors from other condition)
    sum_topk_eigs: sum of top-k eigenvalues of G (same condition as G)
    """
    G = np.asarray(G, dtype=float)
    G = 0.5 * (G + G.T)

    U = np.asarray(U, dtype=float)

    denom = float(sum_topk_eigs)
    if denom <= 0:
        return np.nan

    # numerator = trace(U^T G U)
    num = float(np.trace(U.T @ G @ U))
    return num / denom

In [ ]:
def add_principal_angles_and_elsayed(rdm_out: dict):
    """
    Adds:
      rdm_out["subspace"] = {angles_deg, align_self_by_other, align_other_by_self, align_sym}
    """
    # pull top-k subspaces + normalization constants
    U_self = rdm_out["eig"]["U_self"]
    U_other = rdm_out["eig"]["U_other"]
    sum_topk_self = rdm_out["eig"]["sum_topk_eigs_self"]
    sum_topk_other = rdm_out["eig"]["sum_topk_eigs_other"]

    # pull Grams
    G_self = rdm_out["self"]["G"]
    G_other = rdm_out["other"]["G"]

    # principal angles
    angles_deg = principal_angles_deg(U_self, U_other)

    # directional elsayed indices (match your MATLAB comments)
    # SELF <- OTHER: evaluate G_self in basis U_other
    align_self_by_other = elsayed_alignment_var(G_self, U_other, sum_topk_self)

    # OTHER <- SELF: evaluate G_other in basis U_self
    align_other_by_self = elsayed_alignment_var(G_other, U_self, sum_topk_other)

    align_sym = 0.5 * (align_self_by_other + align_other_by_self)

    rdm_out["subspace"] = dict(
        angles_deg=angles_deg,
        mean_angle_deg=float(np.nanmean(angles_deg)) if angles_deg.size else np.nan,
        align_self_by_other=float(align_self_by_other),
        align_other_by_self=float(align_other_by_self),
        align_sym=float(align_sym),
    )
    return rdm_out

In [ ]:
def add_principal_angles_and_elsayed(rdm_out: dict):
    """
    Adds:
      rdm_out["subspace"] = {angles_deg, align_self_by_other, align_other_by_self, align_sym}
    """
    # pull top-k subspaces + normalization constants
    U_self = rdm_out["eig"]["U_self"]
    U_other = rdm_out["eig"]["U_other"]
    sum_topk_self = rdm_out["eig"]["sum_topk_eigs_self"]
    sum_topk_other = rdm_out["eig"]["sum_topk_eigs_other"]

    # pull Grams
    G_self = rdm_out["self"]["G"]
    G_other = rdm_out["other"]["G"]

    # principal angles
    angles_deg = principal_angles_deg(U_self, U_other)

    # directional elsayed indices (match your MATLAB comments)
    # SELF <- OTHER: evaluate G_self in basis U_other
    align_self_by_other = elsayed_alignment_var(G_self, U_other, sum_topk_self)

    # OTHER <- SELF: evaluate G_other in basis U_self
    align_other_by_self = elsayed_alignment_var(G_other, U_self, sum_topk_other)

    align_sym = 0.5 * (align_self_by_other + align_other_by_self)

    rdm_out["subspace"] = dict(
        angles_deg=angles_deg,
        mean_angle_deg=float(np.nanmean(angles_deg)) if angles_deg.size else np.nan,
        align_self_by_other=float(align_self_by_other),
        align_other_by_self=float(align_other_by_self),
        align_sym=float(align_sym),
    )
    return rdm_out

In [ ]:
rows = []
for pid, regions in rdm_dict.items():
    for region, ro in regions.items():
        attach_eigendecomp_to_rdm_out(ro, frac=0.95, cap=150)
        add_principal_angles_and_elsayed(ro)

        rows.append(dict(
            patient_id=pid,
            region=region,
            n_words=len(ro["words"]),
            k=ro["eig"]["k"],
            eff_self=ro["eig"]["eff_dim_self"],
            eff_other=ro["eig"]["eff_dim_other"],
            mean_angle_deg=ro["subspace"]["mean_angle_deg"],
            align_self_by_other=ro["subspace"]["align_self_by_other"],
            align_other_by_self=ro["subspace"]["align_other_by_self"],
            align_sym=ro["subspace"]["align_sym"],
        ))

summary_df = pd.DataFrame(rows)
summary_df.to_csv('summarydfrdm.csv')

In [ ]:
summary_df.to_csv('summarydfrdm.csv')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_mean_subspace_angle_schematic_v2(
    df,
    region=None,
    angle_col="mean_angle_deg",
    title="Angle between subspaces",
    mean_color="red",
    indiv_color="black",
    indiv_alpha=0.45,
    figsize=(5.2, 4.8),
    save_path=None,
    dpi=300,
):
    d = df.copy()
    if region is not None and "region" in d.columns:
        d = d[d["region"] == region].copy()

    d = d.dropna(subset=[angle_col]).reset_index(drop=True)
    if len(d) == 0:
        raise ValueError("No data to plot after filtering.")

    vals_deg = d[angle_col].to_numpy(dtype=float)
    vals_deg = vals_deg[np.isfinite(vals_deg)]
    mean_deg = np.mean(vals_deg)

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_aspect("equal")
    ax.axis("off")

    # quarter-circle arc
    theta = np.linspace(0, np.pi / 2, 400)
    ax.plot(np.cos(theta), np.sin(theta), color="0.5", lw=1.8)

    # inner arcs
    for r in [1/3, 2/3]:
        ax.plot(r * np.cos(theta), r * np.sin(theta), color="0.8", lw=1.0)

    # radial guides
    for deg in [0, 30, 60, 90]:
        th = np.deg2rad(deg)
        ax.plot([0, np.cos(th)], [0, np.sin(th)], color="0.8", lw=1.0)

    # axes
    ax.plot([0, 1.03], [0, 0], color="black", lw=2.5)
    ax.plot([0, 0], [0, 1.03], color="black", lw=2.5)

    # individual patient rays
    for ang in vals_deg:
        th = np.deg2rad(ang)
        ax.plot(
            [0, 0.94 * np.cos(th)],
            [0, 0.94 * np.sin(th)],
            color=indiv_color,
            lw=1.2,
            alpha=indiv_alpha,
        )

    # mean ray
    th_mean = np.deg2rad(mean_deg)
    ax.plot(
        [0, 0.98 * np.cos(th_mean)],
        [0, 0.98 * np.sin(th_mean)],
        color=mean_color,
        lw=3.0,
    )

    # arrowhead on mean ray
    ax.annotate(
        "",
        xy=(0.98 * np.cos(th_mean), 0.98 * np.sin(th_mean)),
        xytext=(0.84 * np.cos(th_mean), 0.84 * np.sin(th_mean)),
        arrowprops=dict(arrowstyle="-|>", color=mean_color, lw=2, mutation_scale=16),
    )

    # degree labels
    label_r = 1.08
    for deg in [0, 30, 60, 90]:
        th = np.deg2rad(deg)
        x = label_r * np.cos(th)
        y = label_r * np.sin(th)
        ha, va = "center", "center"
        if deg == 0:
            va = "top"
        elif deg == 90:
            ha = "right"
        ax.text(x, y, f"{deg}", fontsize=12, ha=ha, va=va)

    full_title = title + (f"\n{region}" if region is not None else "")
    ax.text(0.0, 1.15, full_title, ha="left", va="bottom", fontsize=16)
    ax.text(0.55, 0.10, f"mean = {mean_deg:.1f}°", color=mean_color, fontsize=12)

    ax.set_xlim(-0.08, 1.16)
    ax.set_ylim(-0.08, 1.2)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
    plt.show()

In [ ]:
import numpy as np
import pandas as pd

rows = []

for pid, regions in rdm_dict.items():
    for region, rdm_out in regions.items():
        angles = np.asarray(rdm_out["subspace"]["angles_deg"], dtype=float)

        rows.append({
            "patient": pid,
            "region": region,
            "mean_angle_deg": np.nanmean(angles),
            "n_angles": len(angles),
        })

df_mean_angles = pd.DataFrame(rows)
df_mean_angles.head()

In [ ]:
plot_mean_subspace_angle_schematic_v2(df_mean_angles, region="hippocampus")

In [ ]:
import numpy as np

all_angles = []
patients = []

for pid, regions in rdm_dict.items():
    if "hippocampus" not in regions:
        continue

    angles = np.asarray(regions["hippocampus"]["subspace"]["angles_deg"], dtype=float)

    all_angles.append(angles)
    patients.append(pid)

len(all_angles)

min_k = min(len(a) for a in all_angles)

A = np.stack([a[:min_k] for a in all_angles])

mean_angles = np.nanmean(A, axis=0)
sem_angles = np.nanstd(A, axis=0) / np.sqrt(A.shape[0])

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(1, len(mean_angles) + 1)

plt.figure(figsize=(5,4))

# individual patients
for i in range(A.shape[0]):
    plt.plot(x, A[i], color="gray", alpha=0.3)

# mean
plt.plot(x, mean_angles, color="red", linewidth=2)

# SEM band
plt.fill_between(
    x,
    mean_angles - sem_angles,
    mean_angles + sem_angles,
    color="red",
    alpha=0.2
)

plt.ylabel("Principal angle (deg)")
plt.xlabel("Subspace dimension")
plt.title("Subspace alignment across patients")

plt.ylim(0, 90)

plt.show()

# reliability of elsayed stability

In [ ]:
# ============================================================
# FULL RDM / SUBSPACE ALIGNMENT PIPELINE (SELF vs OTHER)
# - builds words×neurons matrices from your RSAAnalyzerNotebook.build_FR
# - computes centered cosine Gram (JKJ)
# - eigendecomposition (PSD-guarded) + choose common k (k95 capped)
# - principal angles + Elsayed alignment (directional + symmetric)
# - permutation null: permute OTHER word labels, recompute align_sym
# - split-half ceilings: split NEURONS within each condition, compute Elsayed reliability
# - p-values: (i) cross above-chance vs perm null, (ii) cross below ceilings vs split-halves
# ============================================================

import numpy as np
from joblib import Parallel, delayed
from scipy.linalg import subspace_angles

# ------------------------- helpers -------------------------

def sym(A: np.ndarray) -> np.ndarray:
    A = np.asarray(A, dtype=float)
    return 0.5 * (A + A.T)

def eig_psd(G: np.ndarray, eps: float = 1e-10, sort_desc: bool = True):
    G = sym(G)
    evals, V = np.linalg.eigh(G)     # ascending
    evals = np.where(evals < eps, 0.0, evals)
    if sort_desc:
        idx = np.argsort(evals)[::-1]
        evals, V = evals[idx], V[:, idx]
    return V, evals

def k95_from_evals(evals: np.ndarray, frac: float = 0.95, eps: float = 1e-12) -> int:
    lam = np.asarray(evals, dtype=float)
    tot = lam.sum()
    if tot <= eps:
        return 0
    return int(np.searchsorted(np.cumsum(lam / tot), frac) + 1)

def principal_angles_deg(Ua: np.ndarray, Ub: np.ndarray) -> np.ndarray:
    return np.degrees(subspace_angles(Ua, Ub))

def p_right_tail(x_obs: float, xs: np.ndarray) -> float:
    xs = np.asarray(xs, dtype=float)
    xs = xs[np.isfinite(xs)]
    if xs.size == 0:
        return np.nan
    return float((np.sum(xs >= x_obs) + 1) / (xs.size + 1))

def p_left_tail(x_obs: float, xs: np.ndarray) -> float:
    xs = np.asarray(xs, dtype=float)
    xs = xs[np.isfinite(xs)]
    if xs.size == 0:
        return np.nan
    return float((np.sum(xs <= x_obs) + 1) / (xs.size + 1))

# ------------------------- core math: Gram -------------------------

def centered_cosine_gram_from_raw(X_words_neurons: np.ndarray, eps: float = 1e-12):
    """
    X_words_neurons: (n_words x n_neurons)
    Returns dict {K, G, J} where:
      K = cosine similarity among word patterns
      G = J K J centered Gram
    """
    X = np.asarray(X_words_neurons, dtype=float)
    X = np.nan_to_num(X, nan=0.0)

    norms = np.linalg.norm(X, axis=1, keepdims=True)
    Xn = X / np.maximum(norms, eps)

    K = sym(Xn @ Xn.T)

    n = K.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    G = sym(J @ K @ J)

    return dict(K=K, G=G, J=J)

# ------------------------- Elsayed alignment -------------------------

def elsayed_alignment_var(G: np.ndarray, U: np.ndarray, sum_topk_eigs: float) -> float:
    """
    Directional alignment:
      tr(U^T G U) / sum_{i<=k} eig_i(G)
    """
    denom = float(sum_topk_eigs)
    if denom <= 0:
        return np.nan
    G = sym(G)
    U = np.asarray(U, dtype=float)
    return float(np.trace(U.T @ G @ U) / denom)

# ------------------------- build rdm_out from your analyzer -------------------------

def run_rdm_for_cfg(an, cfg, eps_cos: float = 1e-12):
    """
    an: RSAAnalyzerNotebook instance (has build_FR)
    cfg: RSAConfig
    Returns rdm_out with X_self/X_other as (words x neurons) and centered Gram blocks.
    """
    FR_self, FR_other, words, neuron_ids, shared_counts_df = an.build_FR(cfg)

    # build_FR returns (neurons x words) -> transpose
    X_self = np.asarray(FR_self, dtype=float).T
    X_other = np.asarray(FR_other, dtype=float).T

    rdm_out = dict(
        patient_id=cfg.patient_id,
        region=cfg.region_name,
        words=list(words),
        neuron_ids=list(neuron_ids),
        shared_counts_df=shared_counts_df,
        X_self=X_self,
        X_other=X_other,
        self=centered_cosine_gram_from_raw(X_self, eps=eps_cos),
        other=centered_cosine_gram_from_raw(X_other, eps=eps_cos),
    )
    return rdm_out

# ------------------------- eig + subspace stats -------------------------

def attach_eig_and_subspace(rdm_out: dict, frac: float = 0.95, k_cap: int = 150, eps_eig: float = 1e-10):
    """
    Adds:
      rdm_out["eig"] and rdm_out["subspace"]
    """
    Gs = np.asarray(rdm_out["self"]["G"], dtype=float)
    Go = np.asarray(rdm_out["other"]["G"], dtype=float)

    Vs, es = eig_psd(Gs, eps=eps_eig, sort_desc=True)
    Vo, eo = eig_psd(Go, eps=eps_eig, sort_desc=True)

    k95_s = k95_from_evals(es, frac=frac)
    k95_o = k95_from_evals(eo, frac=frac)
    k = int(max(1, min(k_cap, k95_s, k95_o, len(es), len(eo))))

    U_self = Vs[:, :k]
    U_other = Vo[:, :k]
    sum_self = float(es[:k].sum())
    sum_other = float(eo[:k].sum())

    angles = principal_angles_deg(U_self, U_other)

    a_self_by_other = elsayed_alignment_var(Gs, U_other, sum_self)
    a_other_by_self = elsayed_alignment_var(Go, U_self, sum_other)
    a_sym = 0.5 * (a_self_by_other + a_other_by_self)

    rdm_out["eig"] = dict(
        frac=float(frac),
        k_cap=int(k_cap),
        k95_self=int(k95_s),
        k95_other=int(k95_o),
        k=int(k),
        evals_self=es,
        evals_other=eo,
        U_self=U_self,
        U_other=U_other,
        sum_topk_eigs_self=sum_self,
        sum_topk_eigs_other=sum_other,
    )

    rdm_out["subspace"] = dict(
        k=int(k),
        angles_deg=angles,
        mean_angle_deg=float(np.nanmean(angles)) if angles.size else np.nan,
        align_self_by_other=float(a_self_by_other),
        align_other_by_self=float(a_other_by_self),
        align_sym=float(a_sym),
    )
    return rdm_out

# ------------------------- permutation null: cross alignment above chance -------------------------

def attach_perm_null(rdm_out: dict, n_perm: int = 500, seed: int = 0, eps_eig: float = 1e-10):
    """
    Permute word labels in OTHER (rows/cols of G_other) and recompute align_sym each time.
    Adds rdm_out["subspace"]["perm"].
    """
    rng = np.random.default_rng(seed)

    if "eig" not in rdm_out or "subspace" not in rdm_out:
        attach_eig_and_subspace(rdm_out)

    Gs = np.asarray(rdm_out["self"]["G"], dtype=float)
    Go = np.asarray(rdm_out["other"]["G"], dtype=float)
    n = Gs.shape[0]

    obs = float(rdm_out["subspace"]["align_sym"])
    U_self = np.asarray(rdm_out["eig"]["U_self"], dtype=float)
    k = int(rdm_out["eig"]["k"])
    U_self = U_self[:, :k]
    sum_self = float(rdm_out["eig"]["sum_topk_eigs_self"])

    null = np.zeros(n_perm, dtype=float)
    for i in range(n_perm):
        p = rng.permutation(n)
        Gp = Go[p][:, p]

        Vp, ep = eig_psd(Gp, eps=eps_eig, sort_desc=True)
        U_other_p = Vp[:, :k]
        sum_other_p = float(ep[:k].sum())

        a_self_by_other = elsayed_alignment_var(Gs, U_other_p, sum_self)
        a_other_by_self = elsayed_alignment_var(Gp, U_self, sum_other_p)
        null[i] = 0.5 * (a_self_by_other + a_other_by_self)

    rdm_out["subspace"].setdefault("perm", {})
    rdm_out["subspace"]["perm"] = dict(
        n_perm=int(n_perm),
        seed=int(seed),
        obs=float(obs),
        null=null,
        null_mean=float(np.nanmean(null)),
        null_ci95=tuple(map(float, np.nanpercentile(null, [2.5, 97.5]))),
        p_perm=float(p_right_tail(obs, null)),   # P(null >= obs)
    )
    return rdm_out

# ------------------------- split-half ceilings: within-condition reliability -------------------------

def _one_split_elsayed(X_words_neurons, frac, k_cap, seed, eps_eig, eps_cos):
    rng = np.random.default_rng(seed)
    X = np.asarray(X_words_neurons, dtype=float)
    n_words, n_neur = X.shape
    if n_neur < 2:
        return np.nan

    jidx = rng.permutation(n_neur)
    mid = n_neur // 2
    j1, j2 = jidx[:mid], jidx[mid:]
    if j1.size == 0 or j2.size == 0:
        return np.nan

    G1 = centered_cosine_gram_from_raw(X[:, j1], eps=eps_cos)["G"]
    G2 = centered_cosine_gram_from_raw(X[:, j2], eps=eps_cos)["G"]

    V1, e1 = eig_psd(G1, eps=eps_eig, sort_desc=True)
    V2, e2 = eig_psd(G2, eps=eps_eig, sort_desc=True)

    k1 = k95_from_evals(e1, frac=frac)
    k2 = k95_from_evals(e2, frac=frac)
    k = int(max(1, min(k_cap, k1, k2, len(e1), len(e2))))

    U1, U2 = V1[:, :k], V2[:, :k]
    s1, s2 = float(e1[:k].sum()), float(e2[:k].sum())

    a_1by2 = elsayed_alignment_var(G1, U2, s1)
    a_2by1 = elsayed_alignment_var(G2, U1, s2)
    return 0.5 * (a_1by2 + a_2by1)

def elsayed_splithalf_by_neurons(
    X_words_neurons: np.ndarray,
    n_split: int = 500,
    frac: float = 0.95,
    k_cap: int = 150,
    seed: int = 0,
    eps_eig: float = 1e-10,
    eps_cos: float = 1e-12,
    n_jobs: int = -1,
    backend: str = "loky",
):
    X = np.asarray(X_words_neurons, dtype=float)
    n_words, n_neur = X.shape

    if n_neur < 2:
        aligns = np.full(n_split, np.nan)
        meta = dict(reason="too_few_neurons", n_words=int(n_words), n_neurons=int(n_neur))
        return aligns, meta

    split_seeds = [seed + 10_000 + b for b in range(n_split)]
    aligns = Parallel(n_jobs=n_jobs, backend=backend)(
        delayed(_one_split_elsayed)(X, frac, k_cap, s, eps_eig, eps_cos) for s in split_seeds
    )
    aligns = np.asarray(aligns, dtype=float)

    meta = dict(
        n_split=int(n_split),
        frac=float(frac),
        k_cap=int(k_cap),
        seed=int(seed),
        n_words=int(n_words),
        n_neurons=int(n_neur),
        mean=float(np.nanmean(aligns)),
        median=float(np.nanmedian(aligns)),
        ci95=tuple(map(float, np.nanpercentile(aligns, [2.5, 97.5]))),
    )
    return aligns, meta

def attach_splithalf(rdm_out: dict, n_split=500, frac=0.95, k_cap=150, seed=0, n_jobs=-1, backend="loky",
                     eps_eig=1e-10, eps_cos=1e-12):
    a_self, meta_self = elsayed_splithalf_by_neurons(
        rdm_out["X_self"], n_split=n_split, frac=frac, k_cap=k_cap, seed=seed,
        eps_eig=eps_eig, eps_cos=eps_cos, n_jobs=n_jobs, backend=backend
    )
    a_other, meta_other = elsayed_splithalf_by_neurons(
        rdm_out["X_other"], n_split=n_split, frac=frac, k_cap=k_cap, seed=seed+1,
        eps_eig=eps_eig, eps_cos=eps_cos, n_jobs=n_jobs, backend=backend
    )
    rdm_out["splithalf"] = dict(
        self=dict(aligns=a_self, meta=meta_self),
        other=dict(aligns=a_other, meta=meta_other),
    )
    return rdm_out

# ------------------------- p-values: cross vs ceilings -------------------------

def attach_cross_vs_splithalf_pvals(rdm_out: dict):
    """
    Adds rdm_out["tests"] containing:
      - p_perm: above-chance (from perm null)
      - p_cross_below_self: P(self_split <= cross)
      - p_cross_below_other: P(other_split <= cross)
    """
    if "subspace" not in rdm_out or "align_sym" not in rdm_out["subspace"]:
        raise KeyError("Need rdm_out['subspace']['align_sym'] (run attach_eig_and_subspace).")
    if "splithalf" not in rdm_out:
        raise KeyError("Need rdm_out['splithalf'] (run attach_splithalf).")

    cross = float(rdm_out["subspace"]["align_sym"])
    a_self = np.asarray(rdm_out["splithalf"]["self"]["aligns"], dtype=float)
    a_other = np.asarray(rdm_out["splithalf"]["other"]["aligns"], dtype=float)

    # small p => cross is below ceiling
    p_below_self = p_left_tail(cross, a_self)   # P(split <= cross)
    p_below_other = p_left_tail(cross, a_other)

    # above-chance p from perm block if present
    p_perm = np.nan
    if "perm" in rdm_out.get("subspace", {}):
        p_perm = float(rdm_out["subspace"]["perm"].get("p_perm", np.nan))

    rdm_out["tests"] = dict(
        align_sym_obs=float(cross),

        self_split_mean=float(np.nanmean(a_self)),
        self_split_ci95=tuple(map(float, np.nanpercentile(a_self[np.isfinite(a_self)], [2.5, 97.5]))),
        p_cross_below_self=float(p_below_self),

        other_split_mean=float(np.nanmean(a_other)),
        other_split_ci95=tuple(map(float, np.nanpercentile(a_other[np.isfinite(a_other)], [2.5, 97.5]))),
        p_cross_below_other=float(p_below_other),

        p_perm=float(p_perm),
    )
    return rdm_out


# ------------------------- Procrustes: rotation in neural space -------------------------

def attach_procrustes(rdm_out: dict, k_cap: int = 50, n_perm: int = 500, seed: int = 0, eps: float = 1e-12):
    """
    Orthogonal Procrustes in reduced neural space.
    Projects X_self and X_other onto joint top-k PCA axes, finds optimal
    orthogonal R minimising ||A - B R||_F, reports r2 and permutation p-value.
    r2 = 1 - ||A - B R||_F^2 / ||A||_F^2
    """
    from scipy.linalg import orthogonal_procrustes

    X_self  = np.asarray(rdm_out["X_self"],  dtype=float)
    X_other = np.asarray(rdm_out["X_other"], dtype=float)

    n_words, n_neur = X_self.shape
    k = min(k_cap, n_words - 1, n_neur)

    # joint PCA: shared axes in neuron-space
    X_joint = np.vstack([X_self, X_other])
    X_joint -= X_joint.mean(axis=0)
    _, _, Vt = np.linalg.svd(X_joint, full_matrices=False)
    V_k = Vt[:k].T  # neurons x k

    A = X_self  @ V_k  # words x k
    B = X_other @ V_k

    A -= A.mean(0)
    B -= B.mean(0)

    R, _ = orthogonal_procrustes(B, A)   # minimises ||A - B R||_F
    ss_res = np.sum((A - B @ R) ** 2)
    ss_tot = np.sum(A ** 2)
    r2 = float(1.0 - ss_res / (ss_tot + eps))

    rng = np.random.default_rng(seed)
    null = np.zeros(n_perm, dtype=float)
    for i in range(n_perm):
        Bp = B[rng.permutation(n_words)]
        Rp, _ = orthogonal_procrustes(Bp, A)
        null[i] = 1.0 - np.sum((A - Bp @ Rp) ** 2) / (ss_tot + eps)

    rdm_out["procrustes"] = dict(
        k=int(k),
        r2=r2,
        R=R,
        null_mean=float(np.nanmean(null)),
        null_ci95=tuple(map(float, np.nanpercentile(null, [2.5, 97.5]))),
        p_perm=float(p_right_tail(r2, null)),
        n_perm=int(n_perm),
    )
    return rdm_out

# ------------------------- one-call runner -------------------------

def run_full_rdm_stack(
    an, cfg,
    frac=0.95, k_cap=150,
    n_split=500, split_seed=123,
    n_perm=500, perm_seed=0,
    n_jobs=16, backend="loky",
    eps_cos=1e-12, eps_eig=1e-10,
):
    rdm_out = run_rdm_for_cfg(an, cfg, eps_cos=eps_cos)
    attach_eig_and_subspace(rdm_out, frac=frac, k_cap=k_cap, eps_eig=eps_eig)
    attach_splithalf(rdm_out, n_split=n_split, frac=frac, k_cap=k_cap, seed=split_seed,
                     n_jobs=n_jobs, backend=backend, eps_cos=eps_cos, eps_eig=eps_eig)
    attach_perm_null(rdm_out, n_perm=n_perm, seed=perm_seed, eps_eig=eps_eig)
    attach_cross_vs_splithalf_pvals(rdm_out)
    attach_procrustes(rdm_out)
    return rdm_out

# ============================================================
# HOW TO RUN (per patient / per region)
# ============================================================

# print(rdm_out["patient_id"], rdm_out["region"], rdm_out["tests"])

# If you already have rdm_dict like {pid: {region: rdm_out}} and want to attach everything:
def attach_full_stack_to_rdm_dict(rdm_dict, frac=0.95, k_cap=150, n_split=500, split_seed=123, n_perm=500, perm_seed=0,
                                 n_jobs=16, backend="loky"):
    for pid, regions in rdm_dict.items():
        for region, rdm_out in regions.items():
            attach_eig_and_subspace(rdm_out, frac=frac, k_cap=k_cap)
            attach_splithalf(rdm_out, n_split=n_split, frac=frac, k_cap=k_cap, seed=split_seed,
                             n_jobs=n_jobs, backend=backend)
            attach_perm_null(rdm_out, n_perm=n_perm, seed=perm_seed)
            attach_cross_vs_splithalf_pvals(rdm_out)

            t = rdm_out["tests"]
            print(pid, region,
                  f"align={t['align_sym_obs']:.3f}",
                  f"p_perm={t['p_perm']:.4g}",
                  f"p_below_self={t['p_cross_below_self']:.4g}",
                  f"p_below_other={t['p_cross_below_other']:.4g}")
    return rdm_dict

def attach_all_rdm_stats_inplace(
    rdm_dict: dict,
    frac: float = 0.95,
    k_cap: int = 150,
    n_split: int = 500,
    split_seed: int = 123,
    n_perm: int = 500,
    perm_seed: int = 0,
    n_jobs: int = 16,
    backend: str = "loky",
):
    """
    Mutates rdm_out in-place for each pid/region:
      - eig + subspace (align_sym, angles)
      - split-half ceilings (within self/within other)
      - permutation null (above chance)
      - cross-vs-ceiling p-values
    """
    for pid, regions in rdm_dict.items():
        for region, rdm_out in regions.items():
            # 1) eig + cross subspace stats (creates rdm_out["eig"], rdm_out["subspace"])
            attach_eig_and_subspace(rdm_out, frac=frac, k_cap=k_cap)

            # 2) within-condition split-half ceilings (creates rdm_out["splithalf"])
            attach_splithalf(
                rdm_out,
                n_split=n_split,
                frac=frac,
                k_cap=k_cap,
                seed=split_seed,
                n_jobs=n_jobs,
                backend=backend,
            )

            # 3) above-chance permutation null (creates rdm_out["subspace"]["perm"])
            attach_perm_null(rdm_out, n_perm=n_perm, seed=perm_seed)

            # 4) p-values comparing cross to ceilings + pull p_perm (creates rdm_out["tests"])
            attach_cross_vs_splithalf_pvals(rdm_out)

            # 5) Procrustes: rotation in neural space (creates rdm_out["procrustes"])
            attach_procrustes(rdm_out)

            # quick print
            t = rdm_out["tests"]
            pr = rdm_out.get("procrustes", {})
            print(
                pid, region,
                f"k={rdm_out['eig']['k']}",
                f"align={t['align_sym_obs']:.3f}",
                f"p_perm={t['p_perm']:.3g}",
                f"p_below_self={t['p_cross_below_self']:.3g}",
                f"p_below_other={t['p_cross_below_other']:.3g}",
                f"procrustes_r2={pr.get('r2', float('nan')):.3f}",
                f"p_proc={pr.get('p_perm', float('nan')):.3g}",
            )

    return rdm_dict

In [ ]:
rdm_dict = attach_all_rdm_stats_inplace(
    rdm_dict,
    frac=0.95,
    k_cap=150,
    n_split=500,
    n_perm=500,
    n_jobs=16,
    backend="loky",
)

In [ ]:
def _one_split_elsayed_fixed_k(X_words_neurons, fixed_k, seed, eps_eig, eps_cos):
    rng = np.random.default_rng(seed)
    X = np.asarray(X_words_neurons, dtype=float)
    n_words, n_neur = X.shape
    if n_neur < 2:
        return np.nan

    jidx = rng.permutation(n_neur)
    mid = n_neur // 2
    j1, j2 = jidx[:mid], jidx[mid:]
    if j1.size == 0 or j2.size == 0:
        return np.nan

    G1 = centered_cosine_gram_from_raw(X[:, j1], eps=eps_cos)["G"]
    G2 = centered_cosine_gram_from_raw(X[:, j2], eps=eps_cos)["G"]

    V1, e1 = eig_psd(G1, eps=eps_eig, sort_desc=True)
    V2, e2 = eig_psd(G2, eps=eps_eig, sort_desc=True)

    k_use = int(min(fixed_k, len(e1), len(e2)))
    if k_use < 1:
        return np.nan

    U1, U2 = V1[:, :k_use], V2[:, :k_use]
    s1, s2 = float(e1[:k_use].sum()), float(e2[:k_use].sum())

    a_1by2 = elsayed_alignment_var(G1, U2, s1)
    a_2by1 = elsayed_alignment_var(G2, U1, s2)
    return 0.5 * (a_1by2 + a_2by1)


def elsayed_splithalf_by_neurons_fixed_k(
    X_words_neurons,
    fixed_k,
    n_split=500,
    seed=0,
    eps_eig=1e-10,
    eps_cos=1e-12,
    n_jobs=-1,
    backend="loky",
):
    X = np.asarray(X_words_neurons, dtype=float)
    n_words, n_neur = X.shape

    if n_neur < 2:
        aligns = np.full(n_split, np.nan)
        meta = dict(reason="too_few_neurons", n_words=int(n_words), n_neurons=int(n_neur), fixed_k=int(fixed_k))
        return aligns, meta

    split_seeds = [seed + 10_000 + b for b in range(n_split)]
    aligns = Parallel(n_jobs=n_jobs, backend=backend)(
        delayed(_one_split_elsayed_fixed_k)(X, fixed_k, s, eps_eig, eps_cos)
        for s in split_seeds
    )
    aligns = np.asarray(aligns, dtype=float)

    meta = dict(
        n_split=int(n_split),
        fixed_k=int(fixed_k),
        n_words=int(n_words),
        n_neurons=int(n_neur),
        mean=float(np.nanmean(aligns)),
        median=float(np.nanmedian(aligns)),
        ci95=tuple(map(float, np.nanpercentile(aligns, [2.5, 97.5]))),
    )
    return aligns, meta


def attach_splithalf_fixed_k(
    rdm_out,
    n_split=500,
    seed=0,
    n_jobs=-1,
    backend="loky",
    eps_eig=1e-10,
    eps_cos=1e-12,
):
    fixed_k = int(rdm_out["eig"]["k"])

    a_self, meta_self = elsayed_splithalf_by_neurons_fixed_k(
        rdm_out["X_self"],
        fixed_k=fixed_k,
        n_split=n_split,
        seed=seed,
        eps_eig=eps_eig,
        eps_cos=eps_cos,
        n_jobs=n_jobs,
        backend=backend,
    )
    a_other, meta_other = elsayed_splithalf_by_neurons_fixed_k(
        rdm_out["X_other"],
        fixed_k=fixed_k,
        n_split=n_split,
        seed=seed + 1,
        eps_eig=eps_eig,
        eps_cos=eps_cos,
        n_jobs=n_jobs,
        backend=backend,
    )

    rdm_out["splithalf"] = dict(
        self=dict(aligns=a_self, meta=meta_self),
        other=dict(aligns=a_other, meta=meta_other),
    )
    return rdm_out


def attach_all_rdm_stats_inplace_fixed_k(
    rdm_dict: dict,
    frac: float = 0.95,
    k_cap: int = 150,
    n_split: int = 500,
    split_seed: int = 123,
    n_perm: int = 500,
    perm_seed: int = 0,
    n_jobs: int = 16,
    backend: str = "loky",
    eps_eig: float = 1e-10,
    eps_cos: float = 1e-12,
):
    """
    Same as attach_all_rdm_stats_inplace, but the split-half ceiling
    uses the full-data k stored in rdm_out["eig"]["k"].
    """
    for pid, regions in rdm_dict.items():
        for region, rdm_out in regions.items():
            # 1) eig + cross subspace stats
            attach_eig_and_subspace(
                rdm_out,
                frac=frac,
                k_cap=k_cap,
                eps_eig=eps_eig,
            )

            # 2) within-condition split-half ceilings using FIXED full-data k
            attach_splithalf_fixed_k(
                rdm_out,
                n_split=n_split,
                seed=split_seed,
                n_jobs=n_jobs,
                backend=backend,
                eps_eig=eps_eig,
                eps_cos=eps_cos,
            )

            # 3) above-chance permutation null
            attach_perm_null(
                rdm_out,
                n_perm=n_perm,
                seed=perm_seed,
                eps_eig=eps_eig,
            )

            # 4) p-values comparing cross to ceilings
            attach_cross_vs_splithalf_pvals(rdm_out)

            t = rdm_out["tests"]
            print(
                pid, region,
                f"k={rdm_out['eig']['k']}",
                f"align={t['align_sym_obs']:.3f}",
                f"p_perm={t['p_perm']:.3g}",
                f"p_below_self={t['p_cross_below_self']:.3g}",
                f"p_below_other={t['p_cross_below_other']:.3g}",
            )

    return rdm_dict

In [ ]:
rdm_dict2 = attach_all_rdm_stats_inplace_fixed_k(
    rdm_dict,
    frac=0.95,
    k_cap=150,
    n_split=500,
    n_perm=500,
    n_jobs=16,
    backend="loky",
)

In [ ]:
import numpy as np
from joblib import Parallel, delayed

# ------------------------------------------------------------
# one matched-half cross draw: SELF-half vs OTHER-half
# ------------------------------------------------------------
def _one_matchedhalf_cross_fixed_k(
    X_self_words_neurons,
    X_other_words_neurons,
    fixed_k,
    seed,
    eps_eig=1e-10,
    eps_cos=1e-12,
):
    rng = np.random.default_rng(seed)

    Xs = np.asarray(X_self_words_neurons, dtype=float)
    Xo = np.asarray(X_other_words_neurons, dtype=float)

    n_words_s, n_neur_s = Xs.shape
    n_words_o, n_neur_o = Xo.shape

    if n_words_s != n_words_o:
        raise ValueError(f"SELF and OTHER must have same number of words. Got {n_words_s} vs {n_words_o}.")
    if n_neur_s < 2 or n_neur_o < 2:
        return np.nan

    # split neurons independently within each condition
    js = rng.permutation(n_neur_s)
    jo = rng.permutation(n_neur_o)

    mid_s = n_neur_s // 2
    mid_o = n_neur_o // 2

    js_half = js[:mid_s]
    jo_half = jo[:mid_o]

    if js_half.size == 0 or jo_half.size == 0:
        return np.nan

    Gs = centered_cosine_gram_from_raw(Xs[:, js_half], eps=eps_cos)["G"]
    Go = centered_cosine_gram_from_raw(Xo[:, jo_half], eps=eps_cos)["G"]

    Vs, es = eig_psd(Gs, eps=eps_eig, sort_desc=True)
    Vo, eo = eig_psd(Go, eps=eps_eig, sort_desc=True)

    k_use = int(min(fixed_k, len(es), len(eo)))
    if k_use < 1:
        return np.nan

    U_self = Vs[:, :k_use]
    U_other = Vo[:, :k_use]
    sum_self = float(es[:k_use].sum())
    sum_other = float(eo[:k_use].sum())

    a_self_by_other = elsayed_alignment_var(Gs, U_other, sum_self)
    a_other_by_self = elsayed_alignment_var(Go, U_self, sum_other)
    return 0.5 * (a_self_by_other + a_other_by_self)


# ------------------------------------------------------------
# one matched-half cross null draw: permute OTHER word labels
# ------------------------------------------------------------
def _one_matchedhalf_cross_perm_fixed_k(
    X_self_words_neurons,
    X_other_words_neurons,
    fixed_k,
    seed,
    eps_eig=1e-10,
    eps_cos=1e-12,
):
    rng = np.random.default_rng(seed)

    Xs = np.asarray(X_self_words_neurons, dtype=float)
    Xo = np.asarray(X_other_words_neurons, dtype=float)

    n_words_s, n_neur_s = Xs.shape
    n_words_o, n_neur_o = Xo.shape

    if n_words_s != n_words_o:
        raise ValueError(f"SELF and OTHER must have same number of words. Got {n_words_s} vs {n_words_o}.")
    if n_neur_s < 2 or n_neur_o < 2:
        return np.nan

    js = rng.permutation(n_neur_s)
    jo = rng.permutation(n_neur_o)

    mid_s = n_neur_s // 2
    mid_o = n_neur_o // 2

    js_half = js[:mid_s]
    jo_half = jo[:mid_o]

    if js_half.size == 0 or jo_half.size == 0:
        return np.nan

    Gs = centered_cosine_gram_from_raw(Xs[:, js_half], eps=eps_cos)["G"]
    Go = centered_cosine_gram_from_raw(Xo[:, jo_half], eps=eps_cos)["G"]

    # permute word labels in OTHER half
    p = rng.permutation(n_words_o)
    Go = Go[p][:, p]

    Vs, es = eig_psd(Gs, eps=eps_eig, sort_desc=True)
    Vo, eo = eig_psd(Go, eps=eps_eig, sort_desc=True)

    k_use = int(min(fixed_k, len(es), len(eo)))
    if k_use < 1:
        return np.nan

    U_self = Vs[:, :k_use]
    U_other = Vo[:, :k_use]
    sum_self = float(es[:k_use].sum())
    sum_other = float(eo[:k_use].sum())

    a_self_by_other = elsayed_alignment_var(Gs, U_other, sum_self)
    a_other_by_self = elsayed_alignment_var(Go, U_self, sum_other)
    return 0.5 * (a_self_by_other + a_other_by_self)


# ------------------------------------------------------------
# distributions
# ------------------------------------------------------------
def matchedhalf_cross_alignment_distribution(
    X_self_words_neurons,
    X_other_words_neurons,
    fixed_k,
    n_split=500,
    seed=0,
    eps_eig=1e-10,
    eps_cos=1e-12,
    n_jobs=-1,
    backend="loky",
):
    seeds = [seed + 20_000 + i for i in range(n_split)]
    aligns = Parallel(n_jobs=n_jobs, backend=backend)(
        delayed(_one_matchedhalf_cross_fixed_k)(
            X_self_words_neurons,
            X_other_words_neurons,
            fixed_k,
            s,
            eps_eig,
            eps_cos,
        )
        for s in seeds
    )
    aligns = np.asarray(aligns, dtype=float)

    meta = dict(
        n_split=int(n_split),
        fixed_k=int(fixed_k),
        mean=float(np.nanmean(aligns)),
        median=float(np.nanmedian(aligns)),
        ci95=tuple(map(float, np.nanpercentile(aligns[np.isfinite(aligns)], [2.5, 97.5]))),
    )
    return aligns, meta


def matchedhalf_cross_perm_distribution(
    X_self_words_neurons,
    X_other_words_neurons,
    fixed_k,
    n_perm=500,
    seed=0,
    eps_eig=1e-10,
    eps_cos=1e-12,
    n_jobs=-1,
    backend="loky",
):
    seeds = [seed + 30_000 + i for i in range(n_perm)]
    null = Parallel(n_jobs=n_jobs, backend=backend)(
        delayed(_one_matchedhalf_cross_perm_fixed_k)(
            X_self_words_neurons,
            X_other_words_neurons,
            fixed_k,
            s,
            eps_eig,
            eps_cos,
        )
        for s in seeds
    )
    null = np.asarray(null, dtype=float)

    meta = dict(
        n_perm=int(n_perm),
        fixed_k=int(fixed_k),
        mean=float(np.nanmean(null)),
        median=float(np.nanmedian(null)),
        ci95=tuple(map(float, np.nanpercentile(null[np.isfinite(null)], [2.5, 97.5]))),
    )
    return null, meta


# ------------------------------------------------------------
# attach matched-half analysis to one rdm_out
# ------------------------------------------------------------
def attach_matchedhalf_analysis(
    rdm_out,
    n_split=500,
    n_perm=500,
    seed=0,
    n_jobs=-1,
    backend="loky",
    eps_eig=1e-10,
    eps_cos=1e-12,
):
    fixed_k = int(rdm_out["eig"]["k"])

    # within ceilings, same fixed k
    a_self, meta_self = elsayed_splithalf_by_neurons_fixed_k(
        rdm_out["X_self"],
        fixed_k=fixed_k,
        n_split=n_split,
        seed=seed,
        eps_eig=eps_eig,
        eps_cos=eps_cos,
        n_jobs=n_jobs,
        backend=backend,
    )

    a_other, meta_other = elsayed_splithalf_by_neurons_fixed_k(
        rdm_out["X_other"],
        fixed_k=fixed_k,
        n_split=n_split,
        seed=seed + 1,
        eps_eig=eps_eig,
        eps_cos=eps_cos,
        n_jobs=n_jobs,
        backend=backend,
    )

    # matched-half cross distribution
    a_cross, meta_cross = matchedhalf_cross_alignment_distribution(
        rdm_out["X_self"],
        rdm_out["X_other"],
        fixed_k=fixed_k,
        n_split=n_split,
        seed=seed + 2,
        eps_eig=eps_eig,
        eps_cos=eps_cos,
        n_jobs=n_jobs,
        backend=backend,
    )

    # matched-half permutation null
    a_cross_null, meta_cross_null = matchedhalf_cross_perm_distribution(
        rdm_out["X_self"],
        rdm_out["X_other"],
        fixed_k=fixed_k,
        n_perm=n_perm,
        seed=seed + 3,
        eps_eig=eps_eig,
        eps_cos=eps_cos,
        n_jobs=n_jobs,
        backend=backend,
    )

    # p-values
    cross_mean = float(np.nanmean(a_cross))

    p_cross_above_chance = p_right_tail(cross_mean, a_cross_null)   # P(null >= cross_mean)
    p_cross_below_self = p_left_tail(cross_mean, a_self)            # P(self_within <= cross_mean)
    p_cross_below_other = p_left_tail(cross_mean, a_other)          # P(other_within <= cross_mean)

    rdm_out["matchedhalf"] = dict(
        fixed_k=int(fixed_k),
        self_within=dict(aligns=a_self, meta=meta_self),
        other_within=dict(aligns=a_other, meta=meta_other),
        cross=dict(aligns=a_cross, meta=meta_cross),
        cross_null=dict(aligns=a_cross_null, meta=meta_cross_null),
        tests=dict(
            cross_mean=float(cross_mean),
            self_within_mean=float(np.nanmean(a_self)),
            other_within_mean=float(np.nanmean(a_other)),
            p_cross_above_chance=float(p_cross_above_chance),
            p_cross_below_self=float(p_cross_below_self),
            p_cross_below_other=float(p_cross_below_other),
        ),
    )
    return rdm_out


# ------------------------------------------------------------
# run over whole dict
# ------------------------------------------------------------
def attach_matchedhalf_analysis_inplace(
    rdm_dict,
    frac=0.95,
    k_cap=150,
    n_split=500,
    n_perm=500,
    seed=123,
    n_jobs=16,
    backend="loky",
    eps_eig=1e-10,
    eps_cos=1e-12,
):
    for pid, regions in rdm_dict.items():
        for region, rdm_out in regions.items():
            # make sure full-data k exists
            attach_eig_and_subspace(
                rdm_out,
                frac=frac,
                k_cap=k_cap,
                eps_eig=eps_eig,
            )

            attach_matchedhalf_analysis(
                rdm_out,
                n_split=n_split,
                n_perm=n_perm,
                seed=seed,
                n_jobs=n_jobs,
                backend=backend,
                eps_eig=eps_eig,
                eps_cos=eps_cos,
            )

            t = rdm_out["matchedhalf"]["tests"]
            print(
                pid, region,
                f"k={rdm_out['matchedhalf']['fixed_k']}",
                f"cross_mean={t['cross_mean']:.3f}",
                f"self_within={t['self_within_mean']:.3f}",
                f"other_within={t['other_within_mean']:.3f}",
                f"p_above_chance={t['p_cross_above_chance']:.3g}",
                f"p_below_self={t['p_cross_below_self']:.3g}",
                f"p_below_other={t['p_cross_below_other']:.3g}",
            )

    return rdm_dict

In [ ]:
rdm_dict3 = attach_matchedhalf_analysis_inplace(
    rdm_dict,
    frac=0.95,
    k_cap=150,
    n_split=500,
    n_perm=500,
    seed=123,
    n_jobs=16,
    backend="loky",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional

def p_to_stars(p: float) -> str:
    if p is None or not np.isfinite(p) or p <= 0:
        return ""
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 5e-2: return "*"
    return "n.s."

def _gaussian_kde_1d(x: np.ndarray, grid: np.ndarray, bw: Optional[float] = None) -> np.ndarray:
    """
    Lightweight 1D KDE (no scipy.stats dependency). Returns density on grid.
    """
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size < 2:
        return np.zeros_like(grid)

    # Scott's rule bandwidth if not provided
    if bw is None:
        std = np.std(x, ddof=1)
        bw = (x.size ** (-1 / 5)) * (std if std > 0 else 1.0)
        bw = max(float(bw), 1e-6)

    z = (grid[:, None] - x[None, :]) / bw
    dens = np.exp(-0.5 * z**2).mean(axis=1) / (bw * np.sqrt(2 * np.pi))
    return dens

def plot_alignment_summary_ridge_panel1(
    rdm_out: dict,
    bins: int = 40,
    ridge_bw: Optional[float] = None,
    ridge_scale: float = 0.9,
    title: Optional[str] = None,
    save_path: Optional[str] = None,
    dpi: int = 300,
):
    cross = float(rdm_out["subspace"]["align_sym"])

    a_self = np.asarray(rdm_out["splithalf"]["self"]["aligns"], float)
    a_other = np.asarray(rdm_out["splithalf"]["other"]["aligns"], float)
    a_self = a_self[np.isfinite(a_self)]
    a_other = a_other[np.isfinite(a_other)]

    null = np.asarray(rdm_out["subspace"]["perm"]["null"], float)
    null = null[np.isfinite(null)]
    p_perm = float(rdm_out["subspace"]["perm"]["p_perm"])

    p_below_self = float(rdm_out["tests"]["p_cross_below_self"])
    p_below_other = float(rdm_out["tests"]["p_cross_below_other"])

    all_x = np.concatenate([a_self, a_other, null, np.array([cross])])
    x_lo = float(np.nanpercentile(all_x, 0.5))
    x_hi = float(np.nanpercentile(all_x, 99.5))
    grid = np.linspace(x_lo, x_hi, 400)

    dens_self = _gaussian_kde_1d(a_self, grid, bw=ridge_bw)
    dens_other = _gaussian_kde_1d(a_other, grid, bw=ridge_bw)

    max_d = max(dens_self.max(initial=0.0), dens_other.max(initial=0.0), 1e-12)
    dens_self = dens_self / max_d * ridge_scale
    dens_other = dens_other / max_d * ridge_scale

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    fig.patch.set_facecolor("white")

    if title is None:
        title = f"{rdm_out.get('patient_id','?')} | {rdm_out.get('region','?')}"
    fig.suptitle(title)

    # ===== PANEL 1 =====
    ax = axes[0]
    y_self, y_other = 1.0, 0.0

    ax.fill_between(grid, y_self, y_self + dens_self, alpha=0.5, linewidth=1)
    ax.fill_between(grid, y_other, y_other + dens_other, alpha=0.5, linewidth=1)
    ax.axvline(cross, linewidth=3)

    ax.set_yticks([y_other + 0.15, y_self + 0.15])
    ax.set_yticklabels(["OTHER", "SELF"])
    ax.set_xlabel("Elsayed alignment")
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(-0.2, 1.0 + ridge_scale + 0.2)

    # clean style
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)

    ax.set_title("Within-condition ceilings")

    ax.text(
        cross, 1.0 + ridge_scale + 0.05,
        f"vs SELF: {p_to_stars(p_below_self)}   vs OTHER: {p_to_stars(p_below_other)}",
        ha="center", va="bottom"
    )

    # ===== PANEL 2 =====
    ax = axes[1]

    ax.hist(null, bins=bins, density=True, alpha=0.6)
    ax.axvline(cross, linewidth=3)

    ax.set_xlabel("Elsayed alignment")
    ax.set_ylabel("PDF")
    x2_lo = min(np.min(null), cross)
    x2_hi = max(np.max(null), cross)

    pad = 0.05 * (x2_hi - x2_lo)

    ax.set_xlim(x2_lo - pad, x2_hi + pad)

    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)

    ax.set_title("Permutation null")

    ymax = ax.get_ylim()[1]
    ax.text(cross, ymax * 0.92, f"p_perm: {p_to_stars(p_perm)}",
            ha="center", va="top")

    plt.tight_layout()

    # ===== SAVE AS EPS =====
    if save_path is not None:
        if not save_path.endswith(".eps"):
            save_path = save_path + ".eps"
        plt.savefig(save_path, format="eps", dpi=dpi, bbox_inches="tight")

    plt.show()

In [ ]:
def plot_procrustes(
    rdm_out: dict,
    bins: int = 40,
    title: Optional[str] = None,
    save_path: Optional[str] = None,
    dpi: int = 300,
    max_labels: int = 30,
):
    """
    Two-panel Procrustes visualisation:
      Left  – word scatter in top-2 PCA axes: self (dots) vs rotated-other (crosses),
               lines connecting the same word show residual after rotation.
      Right – permutation null histogram with observed r2 marked.
    """
    from scipy.linalg import orthogonal_procrustes

    pr = rdm_out["procrustes"]
    words = rdm_out.get("words", [])

    X_self  = np.asarray(rdm_out["X_self"],  dtype=float)
    X_other = np.asarray(rdm_out["X_other"], dtype=float)
    n_words, n_neur = X_self.shape
    k = pr["k"]

    # rebuild joint PCA projection (same as attach_procrustes)
    X_joint = np.vstack([X_self, X_other])
    X_joint -= X_joint.mean(axis=0)
    _, _, Vt = np.linalg.svd(X_joint, full_matrices=False)
    V_k = Vt[:k].T

    A = X_self  @ V_k;  A -= A.mean(0)
    B = X_other @ V_k;  B -= B.mean(0)
    R, _ = orthogonal_procrustes(B, A)
    B_rot = B @ R

    r2    = float(pr["r2"])
    null  = np.asarray(pr.get("null_ci95", [np.nan, np.nan]))
    p_perm = float(pr["p_perm"])

    if title is None:
        title = f"{rdm_out.get('patient_id','?')} | {rdm_out.get('region','?')}"

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor("white")
    fig.suptitle(title)

    # ===== PANEL 1: word scatter =====
    ax = axes[0]
    ax.scatter(A[:, 0], A[:, 1], s=50, zorder=3, label="self")
    ax.scatter(B_rot[:, 0], B_rot[:, 1], s=50, marker="x", zorder=3, label="other (rotated)")

    for j in range(n_words):
        ax.plot([A[j, 0], B_rot[j, 0]], [A[j, 1], B_rot[j, 1]],
                color="gray", linewidth=0.7, alpha=0.5, zorder=2)

    if words:
        for j, w in enumerate(words[:max_labels]):
            ax.annotate(w, (A[j, 0], A[j, 1]), fontsize=6,
                        xytext=(3, 3), textcoords="offset points", color="C0", alpha=0.8)

    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.set_title(f"Word geometry: self vs rotated-other\nr² = {r2:.3f}  ({p_to_stars(p_perm)})")
    ax.legend(fontsize=8, frameon=False)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)

    # ===== PANEL 2: permutation null =====
    ax = axes[1]
    null_vals = np.asarray(rdm_out["procrustes"].get("null_mean", np.nan))

    # re-run permutation for plotting if we only stored summary stats
    rng = np.random.default_rng(rdm_out["procrustes"].get("n_perm", 500))
    n_perm = int(rdm_out["procrustes"]["n_perm"])
    eps = 1e-12
    ss_tot = float(np.sum(A ** 2))
    null_dist = np.zeros(n_perm)
    rng2 = np.random.default_rng(0)
    for i in range(n_perm):
        Bp = B[rng2.permutation(n_words)]
        Rp, _ = orthogonal_procrustes(Bp, A)
        null_dist[i] = 1.0 - np.sum((A - Bp @ Rp) ** 2) / (ss_tot + eps)

    ax.hist(null_dist, bins=bins, density=True, alpha=0.6, label="perm null")
    ax.axvline(r2, linewidth=2.5, label=f"observed r²={r2:.3f}")
    ymax = ax.get_ylim()[1]
    ax.text(r2, ymax * 0.92, p_to_stars(p_perm), ha="center", va="top", fontsize=13)
    ax.set_xlabel("Procrustes r²")
    ax.set_ylabel("density")
    ax.set_title("Permutation null")
    ax.legend(fontsize=8, frameon=False)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)

    plt.tight_layout()

    if save_path is not None:
        if not save_path.endswith(".eps"):
            save_path = save_path + ".eps"
        plt.savefig(save_path, format="eps", dpi=dpi, bbox_inches="tight")

    plt.show()


In [ ]:
import os

save_dir = "procrustes_eps"
os.makedirs(save_dir, exist_ok=True)

for pid, regions in rdm_dict.items():
    for region, rdm_out in regions.items():
        if "procrustes" not in rdm_out:
            print(f"skipping {pid} {region} — no procrustes data")
            continue
        print(pid, region)
        save_name = os.path.join(save_dir, f"procrustes_{pid}_{region}.eps")
        plot_procrustes(rdm_out, save_path=save_name)


In [ ]:
import os

save_dir = "elsayed_alignment_eps"
os.makedirs(save_dir, exist_ok=True)

for pid, regions in rdm_dict.items():
    for region, rdm_out in regions.items():

        print(pid, region)

        save_name = os.path.join(
            save_dir,
            f"elsayed_alignment_{pid}_{region}.eps"
        )

        plot_alignment_summary_ridge_panel1(
            rdm_out,
            save_path=save_name
        )

In [ ]:
import numpy as np
import pandas as pd

def mean_finite(x):
    x = np.asarray(x).ravel()
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if x.size else np.nan

rows = []

for pid, regions in rdm_dict.items():
    for region, rdm_out in regions.items():
        sh = rdm_out["splithalf"]
        t  = rdm_out["tests"]

        self_within  = mean_finite(sh["self"]["aligns"])
        other_within = mean_finite(sh["other"]["aligns"])
        within_mean  = float(np.mean([self_within, other_within]))

        across = float(t["align_sym_obs"])  # <-- across-condition alignment (observed)
        k = rdm_out["eig"]["k"]

        rows.append(dict(
            patient=pid,
            region=region,
            k=k,
            self_within=self_within,
            other_within=other_within,
            within_mean=within_mean,
            across=across,
            across_minus_within=across - within_mean,
            across_over_within=across / within_mean if np.isfinite(within_mean) and within_mean != 0 else np.nan,
            p_perm=float(t.get("p_perm", np.nan)),
            p_below_self=float(t.get("p_cross_below_self", np.nan)),
            p_below_other=float(t.get("p_cross_below_other", np.nan)),
        ))

df_align = pd.DataFrame(rows).sort_values(["region", "patient"])
df_align

In [ ]:
import numpy as np
import pandas as pd

def mean_finite(x):
    x = np.asarray(x).ravel()
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if x.size else np.nan

def p_across_below_within_mean(self_aligns, other_aligns, across_obs, pairing="index", rng=0):
    """
    Returns one-sided p = P(across_obs <= within_mean_draw)
    within_mean_draw = 0.5*(self_draw + other_draw)
    pairing:
      - "index": pair ith self with ith other (works if same resampling scheme)
      - "shuffle": randomly pair draws (robust if independent draws)
    """
    a = np.asarray(self_aligns).ravel()
    b = np.asarray(other_aligns).ravel()
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if a.size == 0 or b.size == 0 or not np.isfinite(across_obs):
        return np.nan

    m = min(a.size, b.size)
    a = a[:m]
    b = b[:m]

    if pairing == "shuffle":
        rng = np.random.default_rng(rng)
        b = rng.permutation(b)

    within_mean_draws = 0.5 * (a + b)

    # one-sided: probability that across is BELOW the within-mean ceiling
    return float(np.mean(across_obs <= within_mean_draws))

rows = []

for pid, regions in rdm_dict.items():
    for region, rdm_out in regions.items():
        sh = rdm_out["splithalf"]
        t  = rdm_out["tests"]

        self_aligns  = sh["self"]["aligns"]
        other_aligns = sh["other"]["aligns"]
        across = float(t["align_sym_obs"])
        k = rdm_out["eig"]["k"]

        self_within  = mean_finite(self_aligns)
        other_within = mean_finite(other_aligns)
        within_mean  = float(np.mean([self_within, other_within]))

        p_below_within_mean = p_across_below_within_mean(
            self_aligns, other_aligns, across_obs=across, pairing="index"
        )

        rows.append(dict(
            patient=pid,
            region=region,
            k=k,
            self_within=self_within,
            other_within=other_within,
            within_mean=within_mean,
            across=across,
            across_minus_within=across - within_mean,
            across_over_within=across / within_mean if np.isfinite(within_mean) and within_mean != 0 else np.nan,
            p_perm=float(t.get("p_perm", np.nan)),
            p_below_self=float(t.get("p_cross_below_self", np.nan)),
            p_below_other=float(t.get("p_cross_below_other", np.nan)),
            p_below_within_mean=p_below_within_mean,
        ))

df_align = pd.DataFrame(rows).sort_values(["region", "patient"])
df_align

In [ ]:
df_acrosswithin=df_align[["patient", "within_mean", "across"]]
df_acrosswithin.to_csv('dfacroswith.csv')

# sweep patient

In [ ]:
import copy
import numpy as np
import pandas as pd


def make_fixed_onset_window(shift_ms, length_ms=500, speaker_of_interest="Speaker1"):
    """
    Build a fixed window relative to onset.
    The same dict can be used as window_spec_self or window_spec_other,
    because build_FR will apply it to self and pooled-other separately.
    """
    return dict(
        speaker_of_interest=speaker_of_interest,
        mode="target_vs_other_fixed_window_from_ref",
        target_ref_point="onset",
        target_shift=int(shift_ms),
        target_window_length=int(length_ms),
        other_ref_point="onset",
        other_shift=int(shift_ms),
        other_window_length=int(length_ms),
    )


def run_one_cfg_fixedk_alignment(analyzer, cfg,
                                 frac=0.95,
                                 k_cap=150,
                                 n_split=500,
                                 split_seed=123,
                                 n_perm=500,
                                 perm_seed=0,
                                 n_jobs=16,
                                 backend="loky",
                                 eps_eig=1e-10,
                                 eps_cos=1e-12):
    """
    Runs the fixed-k RDM/subspace pipeline for ONE config.
    Assumes the following helper functions already exist in your notebook:
      - run_rdm_for_cfg
      - attach_eigendecomp_to_rdm_out
      - add_principal_angles_and_elsayed
      - attach_splithalf_fixed_k
      - attach_perm_null
      - attach_cross_vs_splithalf_pvals
    """
    rdm_out = run_rdm_for_cfg(analyzer, cfg)

    attach_eigendecomp_to_rdm_out(
        rdm_out,
        frac=frac,
        cap=k_cap,
        eps=eps_eig,
        store_full_V=False,
    )

    add_principal_angles_and_elsayed(rdm_out)

    attach_splithalf_fixed_k(
        rdm_out,
        n_split=n_split,
        seed=split_seed,
        n_jobs=n_jobs,
        backend=backend,
        eps_eig=eps_eig,
        eps_cos=eps_cos,
    )

    attach_perm_null(
        rdm_out,
        n_perm=n_perm,
        seed=perm_seed,
        eps_eig=eps_eig,
    )

    attach_cross_vs_splithalf_pvals(rdm_out)

    return rdm_out


def summarize_fixedk_result(cfg, rdm_out, self_shift, other_shift, window_len):
    """
    Flatten one result into a row for a summary dataframe.
    """
    eig = rdm_out["eig"]
    sub = rdm_out["subspace"]
    spl = rdm_out["splithalf"]
    tst = rdm_out["tests"]

    self_mean = spl["self"]["meta"].get("mean", np.nan)
    other_mean = spl["other"]["meta"].get("mean", np.nan)

    align_obs = tst.get("align_sym_obs", sub.get("align_sym", np.nan))
    p_perm = tst.get("p_perm", np.nan)
    p_below_self = tst.get("p_cross_below_self", np.nan)
    p_below_other = tst.get("p_cross_below_other", np.nan)

    return dict(
        patient_id=cfg.patient_id,
        region=cfg.region_name,
        self_shift_ms=int(self_shift),
        other_shift_ms=int(other_shift),
        window_len_ms=int(window_len),

        n_words=len(rdm_out["words"]),
        n_neurons=rdm_out["X_self"].shape[1],

        k=int(eig["k"]),
        k95_self=int(eig["k95_self"]),
        k95_other=int(eig["k95_other"]),
        eff_dim_self=float(eig["eff_dim_self"]),
        eff_dim_other=float(eig["eff_dim_other"]),

        mean_angle_deg=float(sub["mean_angle_deg"]),
        align_self_by_other=float(sub["align_self_by_other"]),
        align_other_by_self=float(sub["align_other_by_self"]),
        align_sym=float(sub["align_sym"]),

        splithalf_self_mean=float(self_mean),
        splithalf_other_mean=float(other_mean),
        splithalf_self_ci_lo=float(spl["self"]["meta"]["ci95"][0]),
        splithalf_self_ci_hi=float(spl["self"]["meta"]["ci95"][1]),
        splithalf_other_ci_lo=float(spl["other"]["meta"]["ci95"][0]),
        splithalf_other_ci_hi=float(spl["other"]["meta"]["ci95"][1]),

        p_perm=float(p_perm),
        p_cross_below_self=float(p_below_self),
        p_cross_below_other=float(p_below_other),

        above_chance=bool(np.isfinite(p_perm) and (p_perm < 0.05)),
        below_self=bool(np.isfinite(p_below_self) and (p_below_self < 0.05)),
        below_other=bool(np.isfinite(p_below_other) and (p_below_other < 0.05)),
        below_both=bool(
            np.isfinite(p_below_self) and np.isfinite(p_below_other)
            and (p_below_self < 0.05)
            and (p_below_other < 0.05)
        ),
        target_pattern=bool(
            np.isfinite(p_perm)
            and np.isfinite(p_below_self)
            and np.isfinite(p_below_other)
            and (p_perm < 0.05)
            and (p_below_self < 0.05)
            and (p_below_other < 0.05)
        ),
    )


def sweep_fixedk_alignment_windows(
    analyzer,
    base_configs,
    self_shifts=np.arange(-300, 101, 50),
    other_shifts=np.arange(-300, 101, 50),
    window_len=500,
    frac=0.95,
    k_cap=150,
    n_split=500,
    split_seed=123,
    n_perm=500,
    perm_seed=0,
    n_jobs=16,
    backend="loky",
    save_csv_path=None,
    verbose=True,
):
    """
    Sweep self and other onset-aligned windows.
    For each patient config, evaluates every (self_shift, other_shift) pair.

    self_shifts / other_shifts are in ms relative to onset.
    """
    rows = []
    full_results = {}

    for cfg0 in base_configs:
        pid = cfg0.patient_id
        reg = cfg0.region_name
        full_results.setdefault(pid, {}).setdefault(reg, {})

        for s_shift in self_shifts:
            for o_shift in other_shifts:
                cfg = copy.deepcopy(cfg0)
                cfg.window_spec_self = make_fixed_onset_window(
                    shift_ms=s_shift,
                    length_ms=window_len,
                    speaker_of_interest=cfg.speaker1_col,
                )
                cfg.window_spec_other = make_fixed_onset_window(
                    shift_ms=o_shift,
                    length_ms=window_len,
                    speaker_of_interest=cfg.speaker1_col,
                )

                key = (int(s_shift), int(o_shift), int(window_len))

                try:
                    rdm_out = run_one_cfg_fixedk_alignment(
                        analyzer=analyzer,
                        cfg=cfg,
                        frac=frac,
                        k_cap=k_cap,
                        n_split=n_split,
                        split_seed=split_seed,
                        n_perm=n_perm,
                        perm_seed=perm_seed,
                        n_jobs=n_jobs,
                        backend=backend,
                    )

                    full_results[pid][reg][key] = rdm_out
                    row = summarize_fixedk_result(
                        cfg=cfg,
                        rdm_out=rdm_out,
                        self_shift=s_shift,
                        other_shift=o_shift,
                        window_len=window_len,
                    )

                    rows.append(row)

                    if verbose:
                        print(
                            f"[{pid} | {reg}] "
                            f"self={s_shift:+d} ms, other={o_shift:+d} ms | "
                            f"k={row['k']} | "
                            f"align={row['align_sym']:.3f} | "
                            f"p_perm={row['p_perm']:.3g} | "
                            f"below_self={row['p_cross_below_self']:.3g} | "
                            f"below_other={row['p_cross_below_other']:.3g}"
                        )

                except Exception as e:
                    rows.append(dict(
                        patient_id=pid,
                        region=reg,
                        self_shift_ms=int(s_shift),
                        other_shift_ms=int(o_shift),
                        window_len_ms=int(window_len),
                        error=str(e),
                    ))
                    if verbose:
                        print(f"[{pid} | {reg}] self={s_shift:+d}, other={o_shift:+d} FAILED: {e}")

    summary_df = pd.DataFrame(rows)

    if save_csv_path is not None:
        summary_df.to_csv(save_csv_path, index=False)

    return summary_df, full_results

In [ ]:
self_grid  = np.arange(-300, 101, 50)
other_grid = np.arange(100, 401, 50)

summary_fixedk, full_fixedk = sweep_fixedk_alignment_windows(
    analyzer=an,
    base_configs=configs,
    self_shifts=self_grid,
    other_shifts=other_grid,
    window_len=500,
    frac=0.95,
    k_cap=150,
    n_split=500,
    split_seed=123,
    n_perm=500,
    perm_seed=0,
    n_jobs=16,
    backend="loky",
    save_csv_path="./rdm_fixedk_window_sweep_summary.csv",
    verbose=True,
)

summary_fixedk.head()


In [ ]:
df = summary_fixedk.copy()

if "error" in df.columns:
    df = df[df["error"].isna()]

df_valid = df[
    (df["p_perm"] < 0.05) &
    (df["p_cross_below_self"] < 0.05) &
    (df["p_cross_below_other"] < 0.05)
]
best_per_patient = (
    df_valid
    .sort_values(["patient_id", "align_sym"], ascending=[True, False])
    .groupby("patient_id", as_index=False)
    .first()
)
best_per_patient

def get_best_rdm_outs(best_df, full_fixedk):
    best_rdm = {}

    for _, row in best_df.iterrows():
        pid = row["patient_id"]
        reg = row["region"] if "region" in row else list(full_fixedk[pid].keys())[0]

        key = (
            int(row["self_shift_ms"]),
            int(row["other_shift_ms"]),
            int(row["window_len_ms"]),
        )

        best_rdm[(pid, reg)] = full_fixedk[pid][reg][key]

    return best_rdm

best_rdm_dict = get_best_rdm_outs(best_per_patient, full_fixedk)

In [ ]:
best_per_patient

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional

def p_to_stars(p: float) -> str:
    if p is None or not np.isfinite(p) or p <= 0:
        return ""
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 5e-2: return "*"
    return "n.s."

def _gaussian_kde_1d(x: np.ndarray, grid: np.ndarray, bw: Optional[float] = None) -> np.ndarray:
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size < 2:
        return np.zeros_like(grid)

    if bw is None:
        std = np.std(x, ddof=1)
        bw = (x.size ** (-1 / 5)) * (std if std > 0 else 1.0)
        bw = max(float(bw), 1e-6)

    z = (grid[:, None] - x[None, :]) / bw
    dens = np.exp(-0.5 * z**2).mean(axis=1) / (bw * np.sqrt(2 * np.pi))
    return dens

def _largest_true_run(mask: np.ndarray):
    """
    Return slice for the largest contiguous True segment.
    """
    idx = np.flatnonzero(mask)
    if idx.size == 0:
        return None

    splits = np.where(np.diff(idx) > 1)[0] + 1
    runs = np.split(idx, splits)
    best = max(runs, key=len)
    return slice(best[0], best[-1] + 1)

def _plot_trimmed_ridge(
    ax,
    grid,
    dens,
    y0,
    color=None,
    alpha=0.45,
    lw=1.2,
    support_frac=0.25,
):
    """
    Plot only the main supported portion of the ridge, avoiding long flat tails.
    support_frac is relative to each ridge's own max density.
    """
    dens = np.asarray(dens, float)
    if dens.size == 0 or not np.isfinite(dens).any():
        return

    dmax = np.nanmax(dens)
    if not np.isfinite(dmax) or dmax <= 0:
        return

    mask = dens >= (support_frac * dmax)
    sl = _largest_true_run(mask)
    if sl is None:
        return

    g = grid[sl]
    d = dens[sl]

    ax.fill_between(g, y0, y0 + d, alpha=alpha, linewidth=0, color=color)
    ax.plot(g, y0 + d, linewidth=lw, color=color)

def plot_alignment_summary_single_panel(
    rdm_out: dict,
    ridge_bw: Optional[float] = None,
    ridge_scale: float = 0.95,
    title: Optional[str] = None,
    save_path: Optional[str] = None,
    dpi: int = 300,
    support_frac: float = 0.001,
):
    cross = float(rdm_out["subspace"]["align_sym"])

    a_self = np.asarray(rdm_out["splithalf"]["self"]["aligns"], float)
    a_other = np.asarray(rdm_out["splithalf"]["other"]["aligns"], float)
    a_self = a_self[np.isfinite(a_self)]
    a_other = a_other[np.isfinite(a_other)]

    null = np.asarray(rdm_out["subspace"]["perm"]["null"], float)
    null = null[np.isfinite(null)]

    p_perm = float(rdm_out["subspace"]["perm"]["p_perm"])
    p_below_self = float(rdm_out["tests"]["p_cross_below_self"])
    p_below_other = float(rdm_out["tests"]["p_cross_below_other"])

    all_x = np.concatenate([a_self, a_other, null, np.array([cross])])
    x_lo = float(np.nanpercentile(all_x, 0.5))
    x_hi = float(np.nanpercentile(all_x, 99.5))
    if not np.isfinite(x_lo) or not np.isfinite(x_hi) or x_lo == x_hi:
        x_lo, x_hi = np.nanmin(all_x), np.nanmax(all_x)
        if x_lo == x_hi:
            x_lo -= 1e-3
            x_hi += 1e-3

    grid = np.linspace(x_lo, x_hi, 400)

    dens_null = _gaussian_kde_1d(null, grid, bw=ridge_bw)
    dens_other = _gaussian_kde_1d(a_other, grid, bw=ridge_bw)
    dens_self = _gaussian_kde_1d(a_self, grid, bw=ridge_bw)

    max_d = max(
        dens_null.max(initial=0.0),
        dens_other.max(initial=0.0),
        dens_self.max(initial=0.0),
        1e-12
    )

    dens_null = dens_null / max_d * ridge_scale
    dens_other = dens_other / max_d * ridge_scale
    dens_self = dens_self / max_d * ridge_scale

    if title is None:
        title = f"{rdm_out.get('patient_id', '?')} | {rdm_out.get('region', '?')}"

    fig, ax = plt.subplots(figsize=(7, 4.8))
    fig.patch.set_facecolor("white")

    y_null, y_other, y_self = 0.0, 1.0, 2.0

    _plot_trimmed_ridge(ax, grid, dens_null,  y_null,  alpha=0.45, lw=1.2, support_frac=support_frac)
    _plot_trimmed_ridge(ax, grid, dens_other, y_other, alpha=0.45, lw=1.2, support_frac=support_frac)
    _plot_trimmed_ridge(ax, grid, dens_self,  y_self,  alpha=0.45, lw=1.2, support_frac=support_frac)

    ax.axvline(cross, linewidth=3)

    ax.set_yticks([y_null + 0.12, y_other + 0.12, y_self + 0.12])
    ax.set_yticklabels(["NULL", "OTHER", "SELF"])
    ax.set_xlabel("Elsayed alignment")
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(-0.2, y_self + ridge_scale + 0.45)
    ax.set_title(title)

    stat_text = (
        f"p_perm: {p_to_stars(p_perm)}   "
        f"vs OTHER: {p_to_stars(p_below_other)}   "
        f"vs SELF: {p_to_stars(p_below_self)}"
    )
    ax.text(
        cross,
        y_self + ridge_scale + 0.18,
        stat_text,
        ha="center",
        va="bottom"
    )

    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)

    plt.tight_layout()

    if save_path is not None:
        if not save_path.endswith(".eps"):
            save_path = save_path + ".eps"
        plt.savefig(save_path, format="eps", dpi=dpi, bbox_inches="tight")

    plt.show()

In [ ]:
for (pid, reg), rdm_out in best_rdm_dict.items():

    plot_alignment_summary_single_panel(
        rdm_out,
        title=f"{pid} | {reg} | best window",
        save_path=f"{pid}_{reg}_best_window"
    )

# big patient

In [ ]:
import re
import os
import json
import numpy as np
import pandas as pd
from dataclasses import dataclass
from collections import Counter
from typing import Dict, List, Tuple, Optional, Any

from scipy.stats import pearsonr, spearmanr, norm
from sklearn.metrics.pairwise import cosine_distances

import sys
sys.path.append("/Users/aniluchavez/Documents/Language/Python")
from spike_processing_utils import (
    load_mat_data,
    get_cells_by_region,
    extract_speaker_events,
    compute_spike_sums,
)

# -----------------------------
# basic helpers
# -----------------------------
TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def get_speaker_cols(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r"Speaker\d+", str(c))]
    return sorted(cols, key=lambda x: int(x.replace("Speaker", "")))

def cell_to_first_token(x):
    if pd.isna(x):
        return np.nan
    toks = TOKEN_RE.findall(str(x).strip().lower())
    return toks[0] if toks else np.nan

def words_in_extraction_order(df: pd.DataFrame, speaker_col: str) -> List[Any]:
    tmp = df.copy()
    tmp[speaker_col] = tmp[speaker_col].astype(str).str.strip()
    spk_df = tmp[tmp[speaker_col] != ""]
    return [cell_to_first_token(x) for x in spk_df[speaker_col].values]

def make_shared_counts(words_self: List[str], words_other: List[str]):
    c_self = Counter(words_self)
    c_other = Counter(words_other)
    shared = sorted(set(c_self.keys()) & set(c_other.keys()))
    counts_df = pd.DataFrame({
        "word": shared,
        "n_self": [c_self[w] for w in shared],
        "n_other": [c_other[w] for w in shared],
        "n_total": [c_self[w] + c_other[w] for w in shared],
    }).sort_values(["n_total", "word"], ascending=[False, True]).reset_index(drop=True)
    return shared, counts_df

def word_word_cosine_distance(FR: np.ndarray, col_labels: List[str], fill="col_mean") -> pd.DataFrame:
    X = FR.astype(float).copy()
    if np.isnan(X).any():
        if fill == "col_mean":
            col_means = np.nanmean(X, axis=0, keepdims=True)
            X = np.where(np.isfinite(X), X, col_means)
        elif fill == "zero":
            X = np.nan_to_num(X, nan=0.0)
        else:
            raise ValueError("fill must be 'col_mean' or 'zero'")
    D = cosine_distances(X.T)
    return pd.DataFrame(D, index=col_labels, columns=col_labels)

def vectorize_upper_triangle(D: pd.DataFrame) -> np.ndarray:
    iu = np.triu_indices(D.shape[0], k=1)
    return D.values[iu]

def fisher_required_n_pairs(r_effect: float, alpha: float, power: float) -> int:
    if not (0 < abs(r_effect) < 1):
        return int(1e18)
    z_effect = np.arctanh(r_effect)
    z_alpha = norm.ppf(1 - alpha / 2)
    z_power = norm.ppf(power)
    n_minus_3 = ((z_alpha + z_power) / abs(z_effect)) ** 2
    return int(np.ceil(n_minus_3 + 3))

def geometry_permutation_test(D_self: pd.DataFrame, D_other: pd.DataFrame,
                              n_perm=5000, method="spearman", seed=0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    words = D_self.index.to_numpy()
    iu = np.triu_indices(len(words), k=1)
    v_self = D_self.values[iu]
    perm_stats = np.zeros(n_perm, dtype=float)

    for i in range(n_perm):
        perm = rng.permutation(len(words))
        Dp = D_other.values[perm][:, perm]
        v_other = Dp[iu]
        mask = np.isfinite(v_self) & np.isfinite(v_other)

        if method == "pearson":
            r, _ = pearsonr(v_self[mask], v_other[mask])
        elif method == "spearman":
            r, _ = spearmanr(v_self[mask], v_other[mask])
        else:
            raise ValueError("method must be 'pearson' or 'spearman'")
        perm_stats[i] = r
    return perm_stats

def perm_p_right_tail(r_obs: float, r_perm: np.ndarray) -> float:
    return (np.sum(r_perm >= r_obs) + 1) / (len(r_perm) + 1)

def zscore_rows(X: np.ndarray) -> np.ndarray:
    X = X.astype(float).copy()
    mu = np.nanmean(X, axis=1, keepdims=True)
    sd = np.nanstd(X, axis=1, keepdims=True)
    sd = np.where(sd == 0, 1.0, sd)
    return (X - mu) / sd

# -----------------------------
# plotting
# -----------------------------
import matplotlib.pyplot as plt

def plot_geometry_scatter(v_self, v_other, r_p, r_s, title_prefix="Geometry similarity",
                          save_path=None, show=True):
    plt.figure(figsize=(5, 5))
    plt.scatter(v_self, v_other, s=10, alpha=0.4)
    plt.xlabel("SELF word–word distance")
    plt.ylabel("OTHER word–word distance")
    plt.title(f"{title_prefix}\nPearson r={r_p:.2f}, Spearman ρ={r_s:.2f}")
    try:
        plt.axline((0, 0), slope=1, linestyle="--", linewidth=1)
    except Exception:
        lims = [
            np.nanmin([plt.xlim()[0], plt.ylim()[0]]),
            np.nanmax([plt.xlim()[1], plt.ylim()[1]])
        ]
        plt.plot(lims, lims, linestyle="--", linewidth=1)
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=200)
    if show:
        plt.show()
    plt.close()

def plot_permutation_hist(perm_stats, r_obs, method_label="Spearman",
                          title_prefix="Permutation test: SELF vs OTHER geometry",
                          save_path=None, show=True):
    plt.figure(figsize=(6, 4))
    plt.hist(perm_stats, bins=40, alpha=0.7, label="Null (permuted)")
    plt.axvline(r_obs, linewidth=2, label="Observed")
    plt.xlabel(f"{method_label} geometry correlation")
    plt.ylabel("Count")
    plt.title(title_prefix)
    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=200)
    if show:
        plt.show()
    plt.close()

# -----------------------------
# config
# -----------------------------
@dataclass
class RSAConfig:
    patient_id: str
    excel_path: str
    mat_path: str
    region_ranges: Dict[str, List[Tuple[int, int]]]
    region_name: str

    speaker1_col: str = "Speaker1"
    average_repeats: bool = True
    min_repeats_each_side: int = 1
    bin_mode: str = "explicit_event_bounds"

    window_spec_self: Optional[Dict[str, Any]] = None
    window_spec_other: Optional[Dict[str, Any]] = None

    min_neurons: int = 20
    min_shared_words: int = 20
    power_alpha: float = 0.05
    power_target: float = 0.80
    power_detectable_r: float = 0.20

# -----------------------------
# analyzer
# -----------------------------
class RSAAnalyzerNotebook:
    def __init__(self, n_perm=5000, seed=0, verbose=True):
        self.n_perm = int(n_perm)
        self.seed = int(seed)
        self.verbose = bool(verbose)

    def default_window_spec(self):
        return dict(
            speaker_of_interest="Speaker1",
            mode="target_vs_other_fixed_window_from_ref",
            target_ref_point="onset",
            target_shift=250,
            target_window_length=500,
            other_ref_point="offset",
            other_shift=-500,
            other_window_length=300,
        )

    def shared_word_set(self, df: pd.DataFrame, speaker1_col: str, min_repeats_each_side: int):
        speaker_cols = get_speaker_cols(df)
        other_cols = [c for c in speaker_cols if c != speaker1_col]

        spk1_words = [w for w in words_in_extraction_order(df, speaker1_col) if pd.notna(w)]
        other_words = []
        for spk in other_cols:
            other_words.extend([w for w in words_in_extraction_order(df, spk) if pd.notna(w)])

        c1 = Counter(spk1_words)
        co = Counter(other_words)

        shared = set(c1.keys()) & set(co.keys())
        shared = {w for w in shared if (c1[w] >= min_repeats_each_side and co[w] >= min_repeats_each_side)}
        return shared, speaker_cols

    def get_patient_shared_words(self, cfg: RSAConfig) -> set:
        df = pd.read_excel(cfg.excel_path, keep_default_na=False)
        shared, _ = self.shared_word_set(df, cfg.speaker1_col, cfg.min_repeats_each_side)
        return shared

    def get_global_shared_words(self, configs: List[RSAConfig]) -> List[str]:
        patient_sets = {}
        for cfg in configs:
            shared = self.get_patient_shared_words(cfg)
            patient_sets[cfg.patient_id] = shared
            if self.verbose:
                print(f"[{cfg.patient_id}] within-patient shared words: {len(shared)}")

        if len(patient_sets) == 0:
            return []

        global_words = sorted(set.intersection(*patient_sets.values()))

        if self.verbose:
            print(f"\nGlobal words present in ALL patients: {len(global_words)}")
            print(global_words[:50])

        return global_words

    def build_FR_patient_on_global_words(self, cfg: RSAConfig, global_words: List[str]):
        ws_self = cfg.window_spec_self or self.default_window_spec()
        ws_other = cfg.window_spec_other or self.default_window_spec()

        df = pd.read_excel(cfg.excel_path, keep_default_na=False)
        global_word_set = set(global_words)

        shared_patient, speaker_cols = self.shared_word_set(
            df, cfg.speaker1_col, cfg.min_repeats_each_side
        )
        usable_words = shared_patient & global_word_set
        other_cols = [c for c in speaker_cols if c != cfg.speaker1_col]

        spikes, qual, chan = load_mat_data(cfg.mat_path)
        region_cells_all = get_cells_by_region(chan, qual, cfg.region_ranges)
        if cfg.region_name not in region_cells_all:
            raise ValueError(f"[{cfg.patient_id}] Region '{cfg.region_name}' not found")

        region_cells = {cfg.region_name: region_cells_all[cfg.region_name]}
        n_neurons = len(region_cells_all[cfg.region_name])

        events_self_all = extract_speaker_events(cfg.excel_path, **ws_self)
        events_other_all = extract_speaker_events(cfg.excel_path, **ws_other)

        ev_self = events_self_all.get(cfg.speaker1_col)
        if ev_self is None or len(ev_self) == 0:
            raise ValueError(f"[{cfg.patient_id}] No {cfg.speaker1_col} events")

        words_self_full = words_in_extraction_order(df, cfg.speaker1_col)[:ev_self.shape[0]]
        mask_self = np.array([(w in usable_words) for w in words_self_full], dtype=bool)
        ev_self = ev_self[mask_self]
        words_self = np.array(words_self_full, dtype=object)[mask_self]

        ev_other_list = []
        words_other_list = []
        for spk in other_cols:
            ev = events_other_all.get(spk)
            if ev is None or len(ev) == 0:
                continue

            words_full = words_in_extraction_order(df, spk)[:ev.shape[0]]
            mask = np.array([(w in usable_words) for w in words_full], dtype=bool)

            if np.any(mask):
                ev_other_list.append(ev[mask])
                words_other_list.append(np.array(words_full, dtype=object)[mask])

        if not ev_other_list:
            raise ValueError(f"[{cfg.patient_id}] No usable OTHER events after global-word filtering")

        ev_other = np.vstack(ev_other_list)
        words_other = np.concatenate(words_other_list)

        def compute_fr(ev):
            sums = compute_spike_sums(spikes, region_cells, ev, mode=cfg.bin_mode)
            mat = sums[cfg.region_name]  # [events x neurons]
            win_ms = ev[:, 2] - ev[:, 1]
            win_s = np.where(win_ms > 0, win_ms / 1000.0, np.nan)
            return mat / win_s[:, None]  # [events x neurons]

        fr_self_events = compute_fr(ev_self)
        fr_other_events = compute_fr(ev_other)

        # neurons x words
        FR_self = np.full((n_neurons, len(global_words)), np.nan)
        FR_other = np.full((n_neurons, len(global_words)), np.nan)

        for j, w in enumerate(global_words):
            ii = np.where(words_self == w)[0]
            jj = np.where(words_other == w)[0]

            if ii.size:
                FR_self[:, j] = np.nanmean(fr_self_events[ii, :], axis=0)
            if jj.size:
                FR_other[:, j] = np.nanmean(fr_other_events[jj, :], axis=0)

        word_meta = pd.DataFrame({
            "word": global_words,
            "has_self": [np.any(words_self == w) for w in global_words],
            "has_other": [np.any(words_other == w) for w in global_words],
            "n_self": [int(np.sum(words_self == w)) for w in global_words],
            "n_other": [int(np.sum(words_other == w)) for w in global_words],
        })

        return FR_self, FR_other, word_meta

    def build_super_patient(self, configs: List[RSAConfig], global_words: List[str], zscore_neurons=True):
        FR_self_blocks = []
        FR_other_blocks = []
        neuron_meta_rows = []
        word_meta_rows = []

        for cfg in configs:
            FR_self, FR_other, word_meta = self.build_FR_patient_on_global_words(cfg, global_words)

            if zscore_neurons:
                FR_self = zscore_rows(FR_self)
                FR_other = zscore_rows(FR_other)

            n_neurons = FR_self.shape[0]

            FR_self_blocks.append(FR_self)
            FR_other_blocks.append(FR_other)

            neuron_meta_rows.append(pd.DataFrame({
                "patient_id": [cfg.patient_id] * n_neurons,
                "region": [cfg.region_name] * n_neurons,
                "local_neuron_idx": np.arange(n_neurons),
            }))

            wm = word_meta.copy()
            wm["patient_id"] = cfg.patient_id
            word_meta_rows.append(wm)

            if self.verbose:
                print(f"[{cfg.patient_id}] added {n_neurons} neurons x {len(global_words)} words")

        FR_self_super = np.vstack(FR_self_blocks)
        FR_other_super = np.vstack(FR_other_blocks)
        neuron_meta = pd.concat(neuron_meta_rows, ignore_index=True)
        word_meta_all = pd.concat(word_meta_rows, ignore_index=True)

        return FR_self_super, FR_other_super, neuron_meta, word_meta_all

    def run_super_patient(self, configs: List[RSAConfig], plot=True, show_plots=True,
                          save_figs=False, out_dir="./rsa_outputs_super", zscore_neurons=True):
        global_words = self.get_global_shared_words(configs)

        if len(global_words) == 0:
            raise ValueError("No words found that are shared across all patients.")

        # sanity check
        if self.verbose:
            print("\nChecking that every patient contains all global words...")
            for cfg in configs:
                patient_shared = self.get_patient_shared_words(cfg)
                missing = set(global_words) - set(patient_shared)
                print(f"[{cfg.patient_id}] missing global words: {len(missing)}")

        FR_self_super, FR_other_super, neuron_meta, word_meta_all = self.build_super_patient(
            configs=configs,
            global_words=global_words,
            zscore_neurons=zscore_neurons
        )

        if self.verbose:
            print("\nSuper-patient matrix shapes:")
            print("FR_self_super :", FR_self_super.shape)
            print("FR_other_super:", FR_other_super.shape)

        D_self = word_word_cosine_distance(FR_self_super, global_words)
        D_other = word_word_cosine_distance(FR_other_super, global_words)

        v_self = vectorize_upper_triangle(D_self)
        v_other = vectorize_upper_triangle(D_other)

        mask = np.isfinite(v_self) & np.isfinite(v_other)
        v_self = v_self[mask]
        v_other = v_other[mask]

        r_p, p_p = pearsonr(v_self, v_other)
        r_s, p_s = spearmanr(v_self, v_other)

        perm_p = geometry_permutation_test(D_self, D_other, n_perm=self.n_perm, method="pearson", seed=self.seed)
        perm_s = geometry_permutation_test(D_self, D_other, n_perm=self.n_perm, method="spearman", seed=self.seed)

        perm_pval_p = perm_p_right_tail(r_p, perm_p)
        perm_pval_s = perm_p_right_tail(r_s, perm_s)

        out = dict(
            n_patients=len(configs),
            total_neurons=int(FR_self_super.shape[0]),
            n_global_words=int(len(global_words)),
            n_pairs=int(len(v_self)),
            pearson_r=float(r_p),
            pearson_p=float(p_p),
            spearman_r=float(r_s),
            spearman_p=float(p_s),
            perm_p_pearson=float(perm_pval_p),
            perm_p_spearman=float(perm_pval_s),
        )

        if save_figs and out_dir is not None:
            os.makedirs(out_dir, exist_ok=True)

            pd.DataFrame({"word": global_words}).to_csv(
                os.path.join(out_dir, "global_shared_words.csv"), index=False
            )
            neuron_meta.to_csv(os.path.join(out_dir, "super_patient_neuron_meta.csv"), index=False)
            word_meta_all.to_csv(os.path.join(out_dir, "super_patient_word_meta_all.csv"), index=False)

            np.save(os.path.join(out_dir, "FR_self_super.npy"), FR_self_super)
            np.save(os.path.join(out_dir, "FR_other_super.npy"), FR_other_super)

            D_self.to_csv(os.path.join(out_dir, "D_self_super.csv"))
            D_other.to_csv(os.path.join(out_dir, "D_other_super.csv"))

            pd.DataFrame([out]).to_csv(os.path.join(out_dir, "super_patient_summary.csv"), index=False)

        if plot:
            scatter_path = None if not save_figs else os.path.join(out_dir, "super_patient_scatter.png")
            hist_path = None if not save_figs else os.path.join(out_dir, "super_patient_perm_spearman.png")

            plot_geometry_scatter(
                v_self, v_other, r_p, r_s,
                title_prefix="Super-patient geometry similarity",
                save_path=scatter_path,
                show=show_plots
            )

            plot_permutation_hist(
                perm_s, r_s,
                method_label="Spearman",
                title_prefix="Super-patient permutation test",
                save_path=hist_path,
                show=show_plots
            )

        if self.verbose:
            print(
                f"\nSUPER PATIENT RESULTS\n"
                f"Patients={len(configs)}, total neurons={FR_self_super.shape[0]}, global words={len(global_words)}\n"
                f"Spearman={r_s:.3f} (perm p={perm_pval_s:.3g})\n"
                f"Pearson={r_p:.3f} (perm p={perm_pval_p:.3g})"
            )

        return {
            "summary": out,
            "global_words": global_words,
            "FR_self_super": FR_self_super,
            "FR_other_super": FR_other_super,
            "D_self_super": D_self,
            "D_other_super": D_other,
            "neuron_meta": neuron_meta,
            "word_meta_all": word_meta_all,
            "perm_pearson": perm_p,
            "perm_spearman": perm_s,
        }

In [ ]:
an = RSAAnalyzerNotebook(n_perm=5000, seed=0, verbose=True)

super_out = an.run_super_patient(
    configs=configs,
    plot=True,
    show_plots=True,
    save_figs=True,
    out_dir="./rsa_outputs_super_allpatients",
    zscore_neurons=True,
)

summary_df = pd.DataFrame([super_out["summary"]])
summary_df